# Harmonization B — every storm on the supercell's profile

**What this notebook tests.** The isolated convection and squall line were rerun on the
supercell's profile — 1000 hPa pressure anchoring, a hydrostatically balanced stratosphere,
and the moisture calculation fixed — with the isolated convection's low-level moisture kept
at its intended 11 g/kg and a well-mixed boundary layer imposed the same way as in the
supercell. Only the supercell is unchanged.

**Why it is the right test.** Where Harmonization A equalizes the storms by giving them all
the *same wrong* profile, this one equalizes them by giving them all the *correct* profile.
If the manuscript's cross-storm conclusions survive both, they are not artefacts of the
setup. If they survive only one, that tells you which error was doing the work.

---

**The question this notebook answers.** Not "did the numbers change" — they will. The
question is whether the setup differences bias the three storms *differently*, which is what
would make the manuscript's storm-to-storm comparisons unfair. So the load-bearing output is
not any individual figure but the cross-storm ratio table in §2.7: a setup error that moves
all three storms by the same factor leaves every comparative claim intact, while one that
moves them by different factors does not.

**Structure.**

| Part | Contents | Executed? |
|---|---|---|
| 1 | Initial-state characterization — how large the original setup errors are, and how much each harmonization removes | Yes |
| 2 | Results comparison — the manuscript's cross-storm figures and statistics, rendered once per arm | No — run this yourself |

Part 2 is left unexecuted deliberately: it needs a dask cluster, and it recomputes the
storm-development dataframes for the isolated convection and the squall line, which do not exist on disk yet.

## The two arms

Both `ic_wk` and `sl_wk` are rerun here; `sc_wk` points at the manuscript run in both arms.

The isolated convection uses `correct_wk_wellmixed_qv_fixed`. Note that its RAMSIN is
byte-identical to the one in `correct_wk_wellmixed`, yet the two runs' outputs differ
(surface q_v of 11.015 vs 11.259 g/kg), so one of those RAMSINs no longer records what was
actually run. The `_qv_fixed` output is the one that lands on the intended 11 g/kg, which
is why it is the one used here — and it is a reminder to read the initial state from the
t=0 output rather than from the namelist.

## Imports and parameters

In [ ]:
# Reload edited helper modules automatically before every cell.
# Python caches an imported module in sys.modules, so editing harmonization_utils.py
# mid-session has no effect on an already-started kernel without this. That failure is
# silent rather than loud when the edit adds a keyword argument: the old signature's
# **kwargs swallows it, and a cache hit returns before the argument is ever inspected.
%load_ext autoreload
%autoreload 2

# Native packages
from pathlib import Path
import sys
import warnings

# Other packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML, display
from tqdm.notebook import tqdm

# Locate this notebook's own directory before importing the helper module.
# Jupyter Lab starts the kernel in the notebook's folder, but VS Code usually starts
# it in the workspace root instead, where `harmonization_utils` is not importable --
# so resolving it from the working directory alone fails depending on how you opened
# the notebook. Search the working directory, its parents, and their `validation/`
# subdirectories for the module rather than assuming any one of them.
_candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "validation",
    *Path.cwd().parents,
    *[parent / "validation" for parent in Path.cwd().parents],
]
_validation_dir = next(
    (d for d in _candidate_dirs if (d / "harmonization_utils.py").is_file()),
    None,
)
if _validation_dir is None:
    raise RuntimeError(
        f"Could not find harmonization_utils.py from cwd {Path.cwd()}. Set"
        " _validation_dir by hand to the repo's validation/ directory."
    )
if str(_validation_dir) not in sys.path:
    sys.path.insert(0, str(_validation_dir))
print(f"Using helper module from {_validation_dir}")

# Shared harmonization machinery. Importing this also puts the in-repo `cloudy`
# and `blt_utils_limited` packages on the path, so it must come before them.
import harmonization_utils as hu

# User packages
import cloudy as cm
import blt_utils_limited as blt

plt.style.use(blt.MPL_STYLE)
plt.rcParams["pdf.fonttype"] = 42.0

# Wider tables, since the comparison tables have many columns
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

In [ ]:
# The two arms this notebook compares. "Original" is the set of runs the manuscript
# was written from; "harmonized" is this notebook's harmonization scheme.
ORIGINAL_ARM_LABEL = "original"
HARMONIZED_ARM_LABEL = "harmonized-sc-profile"

RUN_DIRS_BY_ARM = {
    ORIGINAL_ARM_LABEL: hu.PAPER_RUN_DIRS,
    HARMONIZED_ARM_LABEL: hu.HARMONIZED_ON_SC_PROFILE_RUN_DIRS,
}

# Where this notebook writes its figures and figure data. Deliberately not
# blt.FIGS_DIR, so a comparison run can never overwrite a manuscript figure.
OUTPUT_DIR = hu.HARMONIZATION_OUTPUT_ROOT / HARMONIZED_ARM_LABEL
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"
FIGURE_DATA_OUTPUT_DIR = OUTPUT_DIR / "figure_data"
for _output_dir in [FIGURE_OUTPUT_DIR, FIGURE_DATA_OUTPUT_DIR]:
    _output_dir.mkdir(parents=True, exist_ok=True)

# Show which run each storm resolves to in each arm, and flag the ones that differ,
# so the arms are unambiguous before any analysis runs.
_run_dir_rows = []
for _storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    _original_run_dir = RUN_DIRS_BY_ARM[ORIGINAL_ARM_LABEL][_storm_key]
    _harmonized_run_dir = RUN_DIRS_BY_ARM[HARMONIZED_ARM_LABEL][_storm_key]
    _run_dir_rows.append({
        "storm": blt.STORM_LABELS[_storm_key],
        ORIGINAL_ARM_LABEL: _original_run_dir.name,
        HARMONIZED_ARM_LABEL: _harmonized_run_dir.name,
        "rerun": _original_run_dir != _harmonized_run_dir,
    })
display(pd.DataFrame(_run_dir_rows).set_index("storm"))

# Part 1 — Initial-state characterization

Everything here reads the **t=0 analysis file** of each run, never the RAMSIN. That is not
pedantry: one of the RAMSIN/output pairs in this study demonstrably disagrees (see the note
on the harmonized isolated-convection runs above), so the namelist is not a reliable record
of what a run started with. The initial state is horizontally homogeneous, so the
horizontal mean `xarrams` takes is exact rather than an approximation.

## 1.1 Read the t=0 soundings

In [ ]:
# Read the initial sounding for every storm in both arms. Runs shared between arms
# are read once and memoized, so this is cheap despite the apparent duplication.
#
# `calculate_sounding_derived_vars` runs one moist-adiabat parcel integration per
# starting level, which is why the CAPE profile is capped at parcels starting below
# 4 km; the integration itself always runs to the top of the column.
soundings_by_arm = {
    arm_label: hu.read_initial_soundings_for_arm(run_dirs)
    for arm_label, run_dirs in RUN_DIRS_BY_ARM.items()
}

# Relabel with human-readable storm names for plotting.
labelled_soundings_by_arm = {
    arm_label: {
        blt.STORM_LABELS[storm_key]: sounding_df
        for storm_key, sounding_df in soundings.items()
    }
    for arm_label, soundings in soundings_by_arm.items()
}

# Confirm every run sits on the same vertical grid, so the difference tables below
# compare like with like rather than silently interpolating across mismatched levels.
_all_sounding_dfs = [
    df for soundings in soundings_by_arm.values() for df in soundings.values()
]
_reference_heights = _all_sounding_dfs[0]["z"].values
print(
    "All runs share the same vertical grid:",
    all(
        np.allclose(_reference_heights, df["z"].values)
        for df in _all_sounding_dfs
    ),
)

## 1.2 The original setups: how large are the unintended differences?

The squall line and supercell were designed to have **identical** thermodynamics — same
temperature, same moisture, same pressure — differing only in their wind profiles. So every
non-zero entry in the squall-line row below is an unintended difference. The isolated
convection is deliberately drier, so its moisture and CAPE rows are expected to differ; its
pressure and temperature rows are not.

Two height windows are shown. The full column exposes the stratospheric potential-temperature
error; the `z ≤ 10 km` window is the depth the manuscript's analysis actually lives in, and is
the one that matters for whether the results are affected.

In [ ]:
# Difference every original storm against the supercell, over the full column and then
# over the analysis-relevant depth. The `z_of_max_abs_diff_m` column is the load-bearing
# one: a large potential-temperature difference is alarming at 5 km and irrelevant at 23 km,
# and only that column distinguishes the two cases.
for z_max, window_label in [(np.inf, "full column"), (10_000, "z <= 10 km")]:
    display(HTML(f"<h4>Original setups vs the supercell — {window_label}</h4>"))
    display(
        hu.sounding_difference_table(
            labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
            reference_label=blt.STORM_LABELS["sc_wk"],
            z_max=z_max,
        ).round(3)
    )

In [ ]:
# The moisture bug is a few-percent error, invisible on a profile overlay and obvious here.
# Sampled at 100 m: above any surface-layer artefact, well inside the well-mixed boundary
# layer where q_v is constant with height.
display(
    HTML("<h4>Low-level moisture: intended vs actual (original setups)</h4>")
)
display(
    hu.low_level_moisture_table(
        soundings_by_arm[ORIGINAL_ARM_LABEL],
        hu.INTENDED_LOW_LEVEL_MIXING_RATIOS_G_KG,
    ).round(4)
)

In [ ]:
# The pressure-anchoring error specifically. Anchoring one profile at 1000 hPa and another
# at mean sea level leaves potential temperature nearly identical while shifting actual
# temperature at a given height, so theta hides this error and T exposes it.
#
# Reading across the row shows whether the offset is a constant shift (a pure anchoring
# effect) or decays with height (the anchoring difference being absorbed by the lapse rate).
display(
    HTML(
        "<h4>Temperature offset vs the supercell at fixed physical heights"
        " (original)</h4>"
    )
)
display(
    hu.temperature_offset_at_fixed_height_table(
        labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
        reference_label=blt.STORM_LABELS["sc_wk"],
    ).round(4)
)

In [ ]:
# Overlay all three original storms. Reading across a row shows *where* in the column the
# setups disagree; the narrowing height windows separate a stratosphere-only difference
# (visible only in the first column) from one that reaches the storm-bearing layer.
fig, axs = hu.plot_sounding_comparison(
    labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
    suptitle="Original setups — the three storms as the manuscript ran them",
)
cm.plotting.fig_multisave(
    fig, "p1_original_setups_soundings", FIGURE_OUTPUT_DIR
)

## 1.3 The harmonized setups

The same tables for the harmonized on the supercell profile arm. What to look for: the squall-line and supercell
rows should collapse toward zero (they are supposed to be thermodynamically identical), and
the isolated convection's rows should retain only its *intended* moisture difference and the
CAPE that follows from it.

In [ ]:
for z_max, window_label in [(np.inf, "full column"), (10_000, "z <= 10 km")]:
    display(
        HTML(
            f"<h4>{HARMONIZED_ARM_LABEL} vs the supercell — {window_label}</h4>"
        )
    )
    display(
        hu.sounding_difference_table(
            labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
            reference_label=blt.STORM_LABELS["sc_wk"],
            z_max=z_max,
        ).round(3)
    )

In [ ]:
display(
    HTML(
        "<h4>Low-level moisture: intended vs actual"
        f" ({HARMONIZED_ARM_LABEL})</h4>"
    )
)
display(
    hu.low_level_moisture_table(
        soundings_by_arm[HARMONIZED_ARM_LABEL],
        hu.INTENDED_LOW_LEVEL_MIXING_RATIOS_G_KG,
    ).round(4)
)

display(
    HTML(
        "<h4>Temperature offset vs the supercell at fixed heights"
        f" ({HARMONIZED_ARM_LABEL})</h4>"
    )
)
display(
    hu.temperature_offset_at_fixed_height_table(
        labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
        reference_label=blt.STORM_LABELS["sc_wk"],
    ).round(4)
)

In [ ]:
fig, axs = hu.plot_sounding_comparison(
    labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
    suptitle=f"{HARMONIZED_ARM_LABEL} — the three storms after harmonization",
)
cm.plotting.fig_multisave(
    fig, "p1_harmonized_setups_soundings", FIGURE_OUTPUT_DIR
)

## 1.4 The same comparisons on a pressure vertical coordinate

Everything above compares the storms at the same **physical height**, which is what makes the
pressure-anchoring error visible in the first place: anchoring one profile at 1000 hPa and
another at mean sea level shifts temperature by ~1 K at a given height even when the two
soundings are otherwise identical.

Replotting against **pressure** removes that offset by construction — two profiles built from
the same thermodynamic recipe but anchored differently will collapse onto each other here. So
any separation that *survives* this view is a real thermodynamic difference over and above the
anchoring, and any separation that *disappears* was the anchoring alone.

The squall line and supercell are the pair to watch: they were designed to be
thermodynamically identical, so on a pressure axis they should lie on top of one another.
Wherever they do not, something other than the anchoring is also different.

The height windows still filter on `z`, so each panel shows the same set of model levels as
its counterpart above and the two views can be read side by side.

In [ ]:
# Same overlays as sections 1.2 and 1.3, but with pressure as the vertical coordinate.
# The axis is inverted automatically so that low pressure (high altitude) is at the top.
for arm_label in RUN_DIRS_BY_ARM:
    fig, axs = hu.plot_sounding_comparison(
        labelled_soundings_by_arm[arm_label],
        y_variable="PS",
        suptitle=f"{arm_label} — profiles on a pressure vertical coordinate",
    )
    cm.plotting.fig_multisave(
        fig, f"p1_pressure_coordinate_soundings__{arm_label}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# The same pressure-coordinate comparison, differenced first.
#
# This is not a redundant version of the figure above. That one plots absolute profiles, so
# its temperature axis has to span the ~87 K the column actually covers. The differences
# between these setups are a few tenths of a Kelvin through most of the troposphere, and a
# few tenths out of 87 is thinner than the line drawn to represent it. So the overlay reports
# "identical" for a difference worth hundreds of J/kg of CAPE. Differencing first lets the
# axis autoscale to the difference, which is the only way tenths of a Kelvin become visible.
#
# The horizontal scale is deliberately set from the troposphere alone (1000-200 hPa). These
# setups disagree by more than 15 K in the stratosphere, and letting that set the scale would
# flatten the troposphere back into a line.
for arm_label in RUN_DIRS_BY_ARM:
    fig, axs = hu.plot_sounding_difference_profiles(
        labelled_soundings_by_arm[arm_label],
        reference_label=blt.STORM_LABELS["sc_wk"],
        suptitle=(
            f"{arm_label} — each storm minus the supercell, at constant"
            " pressure"
        ),
    )
    cm.plotting.fig_multisave(
        fig, f"p1_pressure_difference_profiles__{arm_label}", FIGURE_OUTPUT_DIR
    )

#### Splitting the surviving temperature difference into its two causes

The ΔT panel above does not fall to zero. It grows steadily from the surface to roughly 1 K
at 200 hPa, and that layer is deep enough to carry a couple of hundred J/kg of CAPE. Two
separate things produce it, and they push in opposite directions:

- **Anchoring.** One profile's surface sits at 1013.25 hPa and the other's at 1000 hPa, so
  the two columns reach any given pressure at different *heights*. Comparing at constant
  pressure therefore samples the same recipe at two different points along itself.
- **A genuinely different θ(z).** The soundings do not describe the same potential-temperature
  profile in the first place. This is the real thermodynamic difference.

Poisson's equation ties T to θ at constant pressure, so the observed difference splits
exactly:

$$\Delta T\big|_p \;=\; \underbrace{\big[\theta_c(z_c) - \theta_r(z_c)\big]\left(\frac{p}{1000\,\mathrm{hPa}}\right)^{R_d/c_p}}_{\theta(z)\ \text{differs}} \;+\; \underbrace{\big[\theta_r(z_c) - \theta_r(z_r)\big]\left(\frac{p}{1000\,\mathrm{hPa}}\right)^{R_d/c_p}}_{\text{anchoring}}$$

where $z_c$ and $z_r$ are the heights at which the two columns reach the same pressure $p$.
The two terms add to the observed difference by construction, so the residual column is a
check on the interpolation rather than a third effect.

The signs are what matter here. If the two terms have opposite signs, the anchoring is
*masking* part of the θ(z) error, and the observed difference is a floor on the real
thermodynamic difference rather than a ceiling.

In [ ]:
# The split, averaged over the free troposphere (850-200 hPa). That window is chosen because
# it carries most of the CAPE integral's weight and excludes the boundary layer, whose own
# moisture differences would otherwise contaminate the answer.
for arm_label in RUN_DIRS_BY_ARM:
    display(
        HTML(
            f"<h4>{arm_label} — ΔT at constant pressure, split by cause"
            " (850-200 hPa mean)</h4>"
        )
    )
    display(
        hu.anchoring_versus_theta_profile_table(
            labelled_soundings_by_arm[arm_label],
            reference_label=blt.STORM_LABELS["sc_wk"],
        ).round(3)
    )

In [ ]:
# Quantify what the pressure view shows. Differencing on a *pressure* grid rather than a
# height grid is the numerical equivalent of the plot above: it removes the anchoring offset,
# so what remains is the thermodynamic difference on top of it.
#
# Compare against the height-based tables in 1.2 and 1.3 -- variables whose max|diff| collapses
# here were being driven by the anchoring alone.
def difference_table_on_pressure_levels(
    labelled_sounding_dfs, reference_label, variables
):
    """Difference each sounding against a reference at matched *pressure* levels.

    Interpolates every sounding onto the reference's pressure levels before
    differencing, so the comparison is at constant pressure rather than constant
    height. np.interp needs its sample points increasing, and pressure decreases
    with height, so both arrays are reversed before interpolating.
    """
    reference_df = labelled_sounding_dfs[reference_label]
    reference_pressures = reference_df["PS"].values

    table_rows = []
    for label, sounding_df in labelled_sounding_dfs.items():
        if label == reference_label:
            continue
        for variable in variables:
            if variable not in sounding_df.columns:
                continue
            interpolated = np.interp(
                reference_pressures[::-1],
                sounding_df["PS"].values[::-1],
                np.asarray(sounding_df[variable].values, dtype=float)[::-1],
            )[::-1]
            difference = interpolated - np.asarray(
                reference_df[variable].values, dtype=float
            )
            finite = np.isfinite(difference)
            if not finite.any():
                continue
            absolute_difference = np.abs(np.where(finite, difference, np.nan))
            worst_ix = int(np.nanargmax(absolute_difference))
            table_rows.append({
                "comparison": label,
                "variable": variable,
                "max_abs_diff": float(np.nanmax(absolute_difference)),
                "p_of_max_abs_diff_hPa": float(reference_pressures[worst_ix]),
                "mean_abs_diff": float(np.nanmean(absolute_difference)),
                "diff_at_surface": float(difference[np.argmax(finite)]),
            })
    return pd.DataFrame(table_rows).set_index(["comparison", "variable"])


# PS is the coordinate now, so differencing it against itself carries no information.
_pressure_view_variables = [
    v for v in hu.SOUNDING_COMPARISON_VARIABLES if v != "PS"
]

for arm_label in RUN_DIRS_BY_ARM:
    display(
        HTML(
            f"<h4>{arm_label} vs the supercell, differenced at constant"
            " pressure</h4>"
        )
    )
    display(
        difference_table_on_pressure_levels(
            labelled_soundings_by_arm[arm_label],
            reference_label=blt.STORM_LABELS["sc_wk"],
            variables=_pressure_view_variables,
        ).round(3)
    )

## 1.5 Skew-T diagrams

Sections 1.2 to 1.4 compare the setups one variable at a time. A skew-T instead shows the
whole thermodynamic state at once, and — more usefully here — it turns CAPE from a number
into an **area**, so a difference in convective potential can be located in the vertical
rather than only quantified.

Three views are drawn, in increasing order of how directly they answer the fairness question.

1. **One panel per setup**, with CAPE and CIN shaded. Rows are arms and columns are storms,
   so reading down a column shows what harmonization did to one storm.
2. **All three storms overlaid**, once per arm. This is the ranking view: in the original arm
   the squall line and supercell should coincide and do not, and after harmonization they
   should.
3. **Each storm's two setups overlaid**, with the *difference* in CAPE shaded directly.

### How the CAPE area is drawn, and why it is not the plain temperature area

Buoyancy is set by density, so CAPE integrates a *virtual* temperature excess, not a plain
one. `metpy.calc.cape_cin` applies that correction internally even though its arguments are
plain temperature and dewpoint, and for these soundings it is worth about 6% of CAPE. The
shading below is therefore bounded by the virtual-temperature curves, which are drawn faintly
alongside the conventional ones. Shading between the plain temperature and parcel curves
instead would look more familiar and would understate every CAPE by that 6%.

On a skew-T the horizontal gap between two curves at fixed pressure is exactly their
temperature difference, and the vertical axis is $-\ln p$. The plotted area between two
curves is therefore proportional to $\int \Delta T \, \mathrm{d}(-\ln p)$, which is the
form CAPE itself takes. That is what makes the third view legitimate: drawing both setups'
positive-area integrands against one shared baseline produces a region whose area is
proportional to the *difference* in their CAPE. Each of those panels prints the difference
twice — once from `cape_cin` and once by integrating the shaded region — so the reader can
confirm that the area really is the number.
### Why the shaded difference is not the area between the two parcel curves

The intuitive reading of a difference panel is that the change in CAPE should be the area
between the two setups' parcel curves. It is not, and the gap is large: for the isolated
convection the parcel-to-parcel area is about $-152$ J/kg against a true change of $-252$ J/kg.

The reason follows from the definition. CAPE integrates the gap between the parcel and its
environment, so a difference of two CAPEs carries two terms:

$$\mathrm{CAPE}_B - \mathrm{CAPE}_A =
\int \left(T_{v,\mathrm{parcel}}^{B} - T_{v,\mathrm{parcel}}^{A}
ight) \mathrm{d}(-\ln p)
\; - \; \int \left(T_{v,\mathrm{env}}^{B} - T_{v,\mathrm{env}}^{A}
ight) \mathrm{d}(-\ln p)$$

The first term is the area between the parcel curves. The second is the area between the
*environment* curves, entering with the opposite sign, because a warmer environment reduces
buoyancy. The two coincide only when both setups share one environment — which is exactly what
a harmonization changes, so they never coincide here.

The shaded band carries both terms at once. That is why it is bounded by one setup's
positive-area edge and by the other setup's edge *transplanted onto the first setup's
environment*. Both of those curves are drawn as thin lines, so the band is bounded by two
things the reader can point at rather than by one visible curve and one implied one.

The table below splits the change into those two terms, plus a remainder for the buoyant layer
changing depth. The three sum to the total exactly. Reading them tells you *why* CAPE moved,
which the total on its own cannot: a colder parcel and a warmer environment are the same number
and different physics.

In [ ]:
# One skew-T per (arm, storm). Rows are arms, columns are storms, and every panel shares
# the same temperature and pressure limits, so the shaded areas are comparable by eye.
# Red is CAPE (the positive area between the LFC and the EL); blue is CIN below the LFC.
fig, skewt_parcel_analyses = hu.plot_skewt_grid(
    labelled_soundings_by_arm,
    suptitle="Initial soundings, CAPE and CIN shaded",
)
cm.plotting.fig_multisave(fig, "p1_skewt_grid_by_arm", FIGURE_OUTPUT_DIR)

In [ ]:
# All three storms on one skew-T, once per arm. The squall line and supercell are the pair
# to watch: they were designed to be thermodynamically identical, so any daylight between
# their curves in the original arm is setup error, and it should close after harmonization.
#
# Each storm's positive area is marked by hatch direction rather than by a tinted fill.
# The three regions overlap almost completely, and overlapping translucent fills blend into
# one patch, which would hide exactly the disagreement the figure is for.
for arm_label, labelled_soundings in labelled_soundings_by_arm.items():
    fig, skew, _ = hu.plot_skewt_overlay(
        labelled_soundings,
        title=f"{arm_label} — all three storms, CAPE hatched",
    )
    cm.plotting.fig_multisave(
        fig, f"p1_skewt_overlay_storms__{arm_label}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# One panel per storm: its original setup against its harmonized setup, with the *change*
# in CAPE shaded. Grey is the original setup's whole positive area, which is there purely
# for scale — a 350 J/kg sliver means something different against 500 J/kg than against 2000.
# Red marks CAPE the harmonization removed, blue marks CAPE it added.
#
# Storms this arm did not rerun are drawn too, unlike in section 1.6. A panel whose shaded
# area is empty and whose ΔCAPE prints as zero is a useful control: it confirms the figure
# machinery reports "no change" when there is genuinely no change.
#
# Everything is drawn at enlarged text, because these panels are the ones most likely to be
# lifted out of the notebook and put in front of an audience. Every text size inside
# `plot_skewt_cape_difference` is a multiple of `font.size`, so raising that one value scales
# the level labels, the caption and the legend together. The three sizes that the project
# style sets in absolute points -- axis titles, axis labels and tick labels -- do not follow
# `font.size`, so they are set here as well.
LARGER_SKEWT_TEXT = {
    "font.size": 20.0,  # drives the level labels, the caption and the legend
    "axes.titlesize": 24.0,
    "axes.labelsize": 22.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
}

# The panel is enlarged along with the text. Scaling the type without scaling the figure just
# leaves less room for the sounding, which is the thing the reader came to look at.
LARGER_SKEWT_FIGSIZE = (11.0, 11.5)

skewt_cape_difference_summaries = []

with plt.rc_context(LARGER_SKEWT_TEXT):
    for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
        storm_label = blt.STORM_LABELS[storm_key]
        fig, skew, difference_summary = hu.plot_skewt_cape_difference(
            ORIGINAL_ARM_LABEL,
            soundings_by_arm[ORIGINAL_ARM_LABEL][storm_key],
            HARMONIZED_ARM_LABEL,
            soundings_by_arm[HARMONIZED_ARM_LABEL][storm_key],
            title=f"{storm_label}: change in CAPE from harmonization",
            figsize=LARGER_SKEWT_FIGSIZE,
        )
        cm.plotting.fig_multisave(
            fig, f"p1_skewt_cape_difference_{storm_key}", FIGURE_OUTPUT_DIR
        )
        difference_summary["storm_label"] = storm_label
        skewt_cape_difference_summaries.append(difference_summary)

# The same numbers in one table.
#
# `from_shaded_area` is the difference recovered by integrating what the figures draw. It is
# kept deliberately: if it ever drifts away from `difference`, the shading has stopped
# meaning what the captions claim.
#
# `parcel_term`, `environment_term` and `layer_change_term` are the split described above,
# and they sum to `difference` exactly. A large `layer_change_term` means the harmonization
# moved the depth of the buoyant layer rather than its strength.
display(HTML("<h4>Surface-based CAPE: harmonized minus original</h4>"))
display(
    hu.skewt_cape_difference_table(skewt_cape_difference_summaries).round(1)
)

## 1.6 What changed for each storm

The tables above compare storms *within* an arm. This section instead compares each storm
against itself across the two arms, which is what tells you how much of a given storm's
result change (in Part 2) can be attributed to its initial state moving at all.

A storm whose two curves lie on top of each other was not rerun, and its Part 2 figures will
be identical between arms by construction.

In [ ]:
# One figure per storm: its original profile against its harmonized profile.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    storm_label = blt.STORM_LABELS[storm_key]
    original_sounding_df = soundings_by_arm[ORIGINAL_ARM_LABEL][storm_key]
    harmonized_sounding_df = soundings_by_arm[HARMONIZED_ARM_LABEL][storm_key]

    # Skip the overlay when the arm did not rerun this storm; an identical pair of
    # curves communicates nothing and just adds figures to scroll past.
    was_rerun = (
        RUN_DIRS_BY_ARM[ORIGINAL_ARM_LABEL][storm_key]
        != RUN_DIRS_BY_ARM[HARMONIZED_ARM_LABEL][storm_key]
    )
    if not was_rerun:
        print(
            f"{storm_label}: not rerun in this harmonization — initial state is"
            " identical"
        )
        continue

    fig, axs = hu.plot_sounding_comparison(
        {
            ORIGINAL_ARM_LABEL: original_sounding_df,
            HARMONIZED_ARM_LABEL: harmonized_sounding_df,
        },
        suptitle=f"{storm_label}: original vs harmonized initial state",
    )
    cm.plotting.fig_multisave(
        fig, f"p1_per_storm_change_{storm_key}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# Quantify the same thing: for each storm, how far did its own initial state move?
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    storm_label = blt.STORM_LABELS[storm_key]
    display(
        HTML(
            f"<h4>{storm_label}: harmonized minus original (z &le; 10 km)</h4>"
        )
    )
    display(
        hu.sounding_difference_table(
            {
                ORIGINAL_ARM_LABEL: soundings_by_arm[ORIGINAL_ARM_LABEL][
                    storm_key
                ],
                HARMONIZED_ARM_LABEL: soundings_by_arm[HARMONIZED_ARM_LABEL][
                    storm_key
                ],
            },
            reference_label=ORIGINAL_ARM_LABEL,
            z_max=10_000,
        ).round(3)
    )

## 1.7 Summary — does harmonization actually equalize the storms?

The single number that matters for fairness is the **spread across storms**, before and
after. Surface-based CAPE is the natural summary: it integrates the temperature, moisture
and pressure errors into one quantity, and it is the quantity most directly tied to how
vigorously each storm can convect.

Because the isolated convection is *meant* to be different, the fair comparison is the
squall line against the supercell — two storms designed to be thermodynamically identical.
Any gap between those two is pure setup error.

In [ ]:
# Surface-based CAPE, CIN and LCL height for every storm in both arms.
stability_rows = []
for arm_label, soundings in soundings_by_arm.items():
    for storm_key, sounding_df in soundings.items():
        stability_rows.append({
            "arm": arm_label,
            "storm": blt.STORM_LABELS[storm_key],
            # Row 0 is the lowest model level, i.e. the surface-based parcel.
            "SBCAPE_J_kg": float(sounding_df["cape"].iloc[0]),
            "SBCIN_J_kg": float(sounding_df["cin"].iloc[0]),
            "LCL_z_m": float(sounding_df["lcl"].iloc[0]),
            "q_v_100m_g_kg": float(
                np.interp(
                    100.0, sounding_df["z"].values, sounding_df["q_v"].values
                )
            ),
        })
stability_table = pd.DataFrame(stability_rows).set_index(["arm", "storm"])
display(HTML("<h4>Surface-based stability by arm</h4>"))
display(stability_table.round(2))

# The fairness number: the squall line and supercell are supposed to be identical, so
# their gap is pure setup error. Track how much each harmonization removes.
display(
    HTML(
        "<h4>Squall line vs supercell — two storms designed to be"
        " identical</h4>"
    )
)
squall_supercell_rows = []
for arm_label in RUN_DIRS_BY_ARM:
    squall_line_cape = stability_table.loc[
        (arm_label, blt.STORM_LABELS["sl_wk"]), "SBCAPE_J_kg"
    ]
    supercell_cape = stability_table.loc[
        (arm_label, blt.STORM_LABELS["sc_wk"]), "SBCAPE_J_kg"
    ]
    squall_supercell_rows.append({
        "arm": arm_label,
        "SL_SBCAPE": squall_line_cape,
        "SC_SBCAPE": supercell_cape,
        "difference_J_kg": squall_line_cape - supercell_cape,
        "percent_of_SC": (
            100.0 * (squall_line_cape - supercell_cape) / supercell_cape
        ),
    })
display(pd.DataFrame(squall_supercell_rows).set_index("arm").round(2))

## 1.8 Winds

In [ ]:
stability_rows = []

for storm_key in blt.STORM_LABELS.values():
    fig, ax = plt.subplots()
    for arm_label, soundings in labelled_soundings_by_arm.items():
        sounding = soundings[storm_key]
        ax.plot(sounding["US"], sounding["z"], label=arm_label)
        ax.set_ylim([None, 5000])
    fig.legend()

# Part 2 — Results comparison

**This part has not been executed.** It needs a dask cluster, and it recomputes the
storm-development dataframe(s) for the isolated convection and the squall line, which are not on disk yet.
Run it yourself and send me any traceback.

### Where the derived products live

The paper runs and the harmonized runs keep their derived products in **different places**:
the paper runs' are in the in-repo tree `replication/data/derived/<storm>/` (what the
committed `blt_utils.py` points the manuscript notebook at), while each harmonized run keeps
its own in `<run_dir>/derived/`. `harmonization_utils` resolves this per run, so nothing here
needs `blt_utils.py` to be in any particular state — but it is worth knowing, because getting
it wrong would not raise, it would just silently recompute everything from raw output.

The precomputed tracer profile datasets exist for **every** run in both arms, so Figures 7
and 9 and the statistics tables need no recomputation. The development dataframes exist only
for the paper runs, so they are computed and cached on first use for the reruns.

Each figure below is rendered **once per arm**, stacked, exactly as
`validation_comparison.ipynb` does — the figure code itself is carried over essentially
verbatim, restructured into functions that take their data as arguments rather than reading
it from a stashed global namespace.

### One important caveat before you run this

Figure 12 and two of the three rows of Figure 11 depend on the **upward mass flux through
10 km**, which the manuscript computes from the 2-second lite files. Every harmonized run
was submitted with `FRQLITE = 0` and therefore wrote **no lite output at all**, so the
manuscript's exact flux is uncomputable for the harmonized arm.

The workaround here is to estimate the flux from the 5-minute analysis output for **both**
arms, using an identical construction, so the arm-to-arm comparison stays valid even though
the absolute values will not match the manuscript's. §2.4.1 quantifies that bias directly by
computing both estimators on the paper runs, which do have lite output. **Read that
validation cell before trusting the Figure 11/12 numbers**, and if the two estimators
disagree badly, treat those two figures as qualitative only.

## 2.0 Cluster and data loading

**Nothing below has to be precalculated.** Every derived product Part 2 uses follows the same
rule: read it from disk if it is there, otherwise compute it from the raw RAMS output and
write it into that run's own `derived/` directory. A run that has only its analysis files
therefore works, and the second pass through this notebook is fast.

Four products are handled that way:

| product | computed by | used for |
|---|---|---|
| in-updraft tracer profiles | `hu.read_or_compute_tracer_totals_datasets` | Figures 7 and 9, entrainment stats |
| development dataframes | `hu.read_or_compute_development_dataframes` | Figure 10, storm centroids |
| upper-level tracer totals | `hu.read_or_cache_dataframe` | Figure 11 |
| coarse upward mass flux | `hu.calculate_cumulative_mass_flux_from_analysis_output` | Figures 11 and 12 |

The first two are the slow ones, because they reduce over the full `(time, z, y, x)` fields.
Budget for a long first pass whenever a run appears here that has not been analysed before.

In [ ]:
# The 5-minute datasets are large enough that the reductions below want a cluster.
USE_DASK_LOCAL_CLUSTER = True

if USE_DASK_LOCAL_CLUSTER:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()
    client = Client(cluster)
    # client = Client(blt.DASK_SCHEDULER_ADDRESS)
    display(client)

In [ ]:
# Load, per arm, the 5-minute analysis output (lazy, dask-backed) and the in-updraft tracer
# profiles.
#
# Nothing here has to be precalculated. Every derived product this notebook needs is read
# from disk when it is already there and computed and cached when it is not, so a run that
# has only its raw RAMS output works. The first pass over such a run is slow, and the second
# is not.
#
# `blt_paths_for_arm` points blt's module-level path constants at this arm's runs for the
# duration of the block and restores them afterwards. That restoration matters -- blt_utils
# is imported by the manuscript notebook in this same repo, and leaving it mutated is how an
# analysis silently reads the wrong run.
storm_dss_by_arm = {}
tracer_totals_profile_dss_by_arm = {}
tracer_totals_dss_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    with hu.blt_paths_for_arm(run_dirs, figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        # 5-minute RAMS output, postprocessed (derived thermodynamics, in_updraft,
        # in_anvil, in_analysis, and the number-concentration tracer variables).
        storm_dss_by_arm[arm_label] = hu.read_storm_datasets(run_dirs)

        # In-updraft tracer vertical profiles, behind Figures 7 and 9 and the entrainment
        # statistics. The paper runs have theirs on disk; a rerun usually does not, so a
        # missing one is computed from the output loaded just above and written into that
        # run's own derived/ directory. That computation touches every gridpoint at every
        # output time, so budget for it the first time a new run appears here.
        (
            tracer_totals_profile_dss_by_arm[arm_label],
            tracer_totals_dss_by_arm[arm_label],
        ) = hu.read_or_compute_tracer_totals_datasets(
            run_dirs, storm_dss_by_arm[arm_label]
        )

# Output interval and grid spacing, which the flux estimator needs. Uniform across runs.
# Defined here rather than after the expensive step below, so that every later cell still
# has them even if the development-dataframe computation fails.
_any_storm_ds = next(iter(storm_dss_by_arm[ORIGINAL_ARM_LABEL].values()))
DELTA_T = (
    _any_storm_ds["time"].values[1] - _any_storm_ds["time"].values[0]
) / np.timedelta64(1, "s")
DELTA_X = _any_storm_ds["x"].values[1] - _any_storm_ds["x"].values[0]
print(f"\nDELTA_T = {DELTA_T} s, DELTA_X = {DELTA_X} m")

### Development dataframes

Separated from the load above because it is one of the two expensive steps in Part 2: it
reduces over the full `(time, z, y, x)` updraft, anvil and rainfall fields. The paper runs
have theirs cached already; anything else is computed here and written into that run's own
`derived/` directory, so a second pass through this notebook is fast.

This cell also attaches each storm's centroid position onto its dataset, which is what lets
the later figures take cross sections that follow the storm.

In [ ]:
# Development dataframes. Read from disk where they exist, computed and cached where they
# do not, so no run has to be preprocessed before it can be analysed here.
development_dfs_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    with hu.blt_paths_for_arm(run_dirs, figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        development_dfs_by_arm[arm_label] = (
            hu.read_or_compute_development_dataframes(
                run_dirs, storm_dss_by_arm[arm_label]
            )
        )

display(development_dfs_by_arm[HARMONIZED_ARM_LABEL]["sc_wk"].head())

## 2.1 Figure 7 — tracer entrainment

Entrained fraction of cumulatively emitted tracer (top block) and mean tracer mixing ratio
(bottom block), for fixed-source and rain-sourced tracers in each storm.

In [ ]:
def plot_figure_7_tracer_entrainment(arm_label, tracer_totals_dss):
    """Render Figure 7 for one arm.

    Carried over from `validation_comparison.ipynb` essentially verbatim; the only
    structural change is that the tracer datasets arrive as an argument instead of
    being restored from a stashed global namespace, and the figure name carries the
    arm label so the two arms write distinct files.
    """
    figure_name = f"f07_tracer_entrainment__{arm_label}"
    figure_data = {}

    fig = plt.figure(figsize=(12, 11))
    outer_subfigs = fig.subfigures(nrows=2, hspace=0.05)

    ########## Fractions ##########
    axs = outer_subfigs[0].subplots(nrows=2, ncols=3, sharex=True)
    outer_subfigs[0].subplots_adjust(
        left=0.13, hspace=0.5, top=0.87, right=0.82
    )

    for tracer_type_ix, tracer_type in enumerate(
        ["Fixed-source", "Rain-sourced"]
    ):
        for storm_ix, (storm, this_tracer_ds) in enumerate(
            tracer_totals_dss.items()
        ):
            ax = axs[tracer_type_ix, storm_ix]
            if tracer_type_ix == 0:
                for ix, tracer in enumerate(blt.FS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_frac_entrained_of_emitted",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        this_tracer_ds[tracer + "_nc"]
                        / this_tracer_ds["ACC" + tracer + "_nc"],
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                            ix / len(blt.FS_TRACERS)
                        ),
                    )
            else:
                for ix, tracer in enumerate(blt.RS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_frac_entrained_of_emitted",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        (
                            this_tracer_ds[tracer + "_nc"]
                            / this_tracer_ds["ACC" + tracer + "_nc"]
                        ).values,
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                            ix / len(blt.RS_TRACERS)
                        ),
                        label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[ix])
                        + " mm/hr",
                    )

            ax.set_title(
                blt.axes_title(
                    storm_ix + tracer_type_ix * 3, blt.STORM_LABELS[storm]
                )
            )
            if storm_ix == 0:
                ax.set_ylabel("Fraction")
            blt.label_x_axis(ax)

        # Share y on the mean row; x is already shared across the subfigure.
        cm.plotting.share_axes(axs[1], x=False, y=True)

    outer_subfigs[0].suptitle(
        "Entrained fraction of cumulatively emitted tracer", fontsize=22
    )

    ########## Means ##########
    axs = outer_subfigs[1].subplots(nrows=2, ncols=3, sharex=True, sharey="row")
    outer_subfigs[1].subplots_adjust(
        left=0.15, hspace=0.5, top=0.87, right=0.82
    )

    for tracer_type_ix, tracer_type in enumerate(
        ["Fixed-source", "Rain-sourced"]
    ):
        for storm_ix, (storm, this_tracer_ds) in enumerate(
            tracer_totals_dss.items()
        ):
            ax = axs[tracer_type_ix, storm_ix]
            if tracer_type_ix == 0:
                for ix, tracer in enumerate(sorted(blt.FS_TRACERS)):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_nc",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        this_tracer_ds[tracer + "_nc"]
                        / this_tracer_ds["air_mass"],
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                            ix / len(blt.FS_TRACERS)
                        ),
                    )
            else:
                for ix, tracer in enumerate(blt.RS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_nc",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        (
                            this_tracer_ds[tracer + "_nc"]
                            / this_tracer_ds["air_mass"]
                        ).values,
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                            ix / len(blt.RS_TRACERS)
                        ),
                        label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[ix])
                        + " mm/hr",
                    )

            # Rain-sourced mixing ratios span orders of magnitude
            if tracer_type_ix == 1:
                ax.set_yscale("log")
                ax.set_ylim([5e2, 2e5])
            ax.set_title(
                blt.axes_title(
                    6 + storm_ix + tracer_type_ix * 3, blt.STORM_LABELS[storm]
                )
            )
            if storm_ix == 0:
                ax.set_ylabel("#/kg")
            blt.label_x_axis(ax)

    outer_subfigs[1].suptitle("Mean tracer mixing ratio", fontsize=22)

    # Colorbars: emission time for the fixed-source tracers, a legend for rain-sourced
    for subfig in outer_subfigs:
        fs_cbar_ax = subfig.add_axes(
            [0.83, 0.55, 0.02, 0.32]
        )  # Left, bottom, width, height
        fs_cbar = mpl.colorbar.ColorbarBase(
            ax=fs_cbar_ax,
            cmap=blt.PROGRESSIVE_CMAP + "_r",
            norm=mpl.colors.Normalize(vmin=0, vmax=1),
            orientation="vertical",
        )
        # Ticks are a linear range over the 0-1 normalized emission time
        tick_labels = np.arange(0, 181, 30)
        tick_values = (tick_labels / 180)[::-1]
        fs_cbar.set_ticks(tick_values)
        fs_cbar.set_ticklabels(tick_labels)
        fs_cbar_ax.xaxis.set_label_position("top")
        fs_cbar_ax.set_xlabel("Emission time\n(minutes)")
        fs_cbar_ax.get_xaxis().labelpad = 10

        cm.plotting.clean_legend(
            subfig.axes[3],
            bbox_transform=subfig.transSubfigure,
            title="Rain threshold",
            bbox_to_anchor=[0.800, 0.468],
            fontsize=14,
            title_fontsize=16,
            loc="upper left",
        )

    # Row headers
    row_headers = {}
    for subfig_ix, subfig in enumerate(outer_subfigs):
        for row_ix in range(2):
            row_headers[(subfig_ix, row_ix)] = cm.plotting.add_row_header(
                subfig.axes[row_ix * 3],
                "Fixed-source" if row_ix == 0 else "Rain-sourced",
            )

    # The log scale on the second subfigure's lower row shifts its header; pull it back
    # into line with the header above it.
    reference_header = row_headers[(1, 0)]
    problematic_header = row_headers[(1, 1)]
    problematic_header.set_position((
        reference_header.get_position()[0],
        problematic_header.get_position()[1],
    ))

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(
        fig, figure_name, FIGURE_OUTPUT_DIR, no_title_version=False
    )
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        plot_figure_7_tracer_entrainment(
            arm_label, tracer_totals_dss_by_arm[arm_label]
        )

## 2.2 Figure 9 — rain-sourced tracer entrainment factors

The decomposition of rain-sourced entrainment into its factors: fixed-source mixing ratio
and updraft mass, rain-sourced fraction, rain production efficiency, and rain-sourced
entrainment efficiency. This is the figure that carries most of the manuscript's mechanistic
cross-storm argument, so it is the one most worth scrutinizing between arms.

In [ ]:
def plot_figure_9_rs_entrainment_factors(arm_label, tracer_totals_dss):
    """Render Figure 9 for one arm. Body carried over verbatim from the template."""
    figure_name = f"f09_rs_entrainment_factors__{arm_label}"
    figure_data = {}

    fig, axs = plt.subplots(
        nrows=4, ncols=3, figsize=(12, 11), sharex=True, sharey="row"
    )
    fig.subplots_adjust(wspace=0.55, left=0.19, hspace=0.5, bottom=0)

    headers = []
    bbox_to_anchor = (0.5, -0.23)

    # Each row is its own function so it can be broken out into a standalone figure
    def rs_factors_fs_entrainment(axs, ax_title_start_ix=0):
        updraft_axs = []
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(
            tracer_totals_dss.items()
        ):
            ax = axs[storm_ix]
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_fs_entrainment",
                ax.plot,
                this_tracer_totals_ds["t_minutes"],
                this_tracer_totals_ds["total_fs_tracer_nc"]
                / this_tracer_totals_ds["air_mass"],
                label=r"$\overline{q_{RS}}$",
                color=cm.plotting.get_nth_color(0),
            )
            # Empty line so the legend picks up the twin axis's series too
            ax.plot(
                [], [], color=cm.plotting.get_nth_color(1), label="Updraft mass"
            )
            updraft_ax = ax.twinx()
            updraft_axs.append(updraft_ax)
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_updraft_mass",
                updraft_ax.plot,
                this_tracer_totals_ds["t_minutes"],
                this_tracer_totals_ds["air_mass"],
                color=cm.plotting.get_nth_color(1),
            )
            updraft_ax.set_yscale("log")
            ax.set_title(
                blt.axes_title(
                    ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]
                )
            )
            ax.set_ylabel(r"$\#\ kg^{-1}$")
            blt.label_x_axis(ax)
            updraft_ax.set_ylabel("kg")

        cm.plotting.share_axes(updraft_axs, x=False, y=True)
        for ax in updraft_axs:
            ax.set_ylim([1e9, 2e14])
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=2)

    def rs_factors_rs_fraction(axs, ax_title_start_ix=0):
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(
            tracer_totals_dss.items()
        ):
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rs_fraction",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds[tracer + "_nc"]
                    / this_tracer_totals_ds["total_fs_tracer_nc"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix])
                    + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                        tracer_ix / len(blt.RS_TRACERS)
                    ),
                )
            ax.set_title(
                blt.axes_title(
                    ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]
                )
            )
            if storm_ix == 0:
                ax.set_ylabel("Fraction")
            blt.label_x_axis(ax)
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=5)

    def rs_factors_rain_efficiency(axs, ax_title_start_ix=0, storms=None):
        storms = storms if storms is not None else tracer_totals_dss.keys()
        for storm_ix, storm in enumerate(storms):
            this_tracer_totals_ds = tracer_totals_dss[storm]
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rain_efficiency",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds["ACC" + tracer + "_nc"]
                    / this_tracer_totals_ds["air_mass"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix])
                    + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                        tracer_ix / len(blt.RS_TRACERS)
                    ),
                )
                ax.set_title(
                    blt.axes_title(
                        ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]
                    )
                )
                if storm_ix == 0:
                    ax.set_ylabel(r"$\#\ kg^{-1}$")
            ax.set_yscale("log")
            blt.label_x_axis(ax)
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=5)

    def rs_factors_rs_entrainment_efficiency(axs, ax_title_start_ix=0):
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(
            tracer_totals_dss.items()
        ):
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rs_entrainment_efficiency",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds[tracer + "_nc"]
                    / this_tracer_totals_ds["ACC" + tracer + "_nc"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix])
                    + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                        tracer_ix / len(blt.RS_TRACERS)
                    ),
                )
                ax.set_title(
                    blt.axes_title(9 + storm_ix, blt.STORM_LABELS[storm])
                )
                if storm_ix == 0:
                    ax.set_ylabel("Fraction")
                blt.label_x_axis(ax)
        axs[1].legend(
            loc="center",
            bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1] - 0.1),
            ncol=5,
        )

    rs_factors_fs_entrainment(axs[0], ax_title_start_ix=0)
    rs_factors_rs_fraction(axs[1], ax_title_start_ix=3)
    rs_factors_rain_efficiency(axs[2], ax_title_start_ix=6)
    rs_factors_rs_entrainment_efficiency(axs[3], ax_title_start_ix=9)

    header_pad = 22
    headers.append(
        cm.plotting.add_row_header(
            axs[0][0],
            "FS tracer mixing\n"
            r"ratio $(\overline{q_{FS}})$ and"
            "\nupdraft mass "
            r" ($M$)",
            pad=header_pad,
            xycoords="subfigure fraction",
            xy=(0.09, 0.87),
            fontsize=22,
        )
    )
    headers.append(
        cm.plotting.add_row_header(
            axs[1][0],
            "RS tracer\n" r"fraction $(f_{RS})$",
            pad=header_pad,
            xycoords="subfigure fraction",
            xy=(0.09, 0.63),
            fontsize=22,
        )
    )
    headers.append(
        cm.plotting.add_row_header(
            axs[2][0],
            "Rain production\n" r"efficiency $(\eta_{RP})$",
            pad=header_pad,
            xycoords="subfigure fraction",
            xy=(0.09, 0.38),
            fontsize=22,
        )
    )
    headers.append(
        cm.plotting.add_row_header(
            axs[3][0],
            "RS tracer entrainment\n" r"efficiency $(\eta_{RSE}$)",
            pad=header_pad,
            xycoords="subfigure fraction",
            xy=(0.09, 0.14),
            fontsize=22,
        )
    )

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(fig, figure_name, FIGURE_OUTPUT_DIR)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        plot_figure_9_rs_entrainment_factors(
            arm_label, tracer_totals_dss_by_arm[arm_label]
        )

## 2.3 Figure 10 — storm development statistics

Max updraft speed and updraft mass, and rainfall area by threshold. This is the figure most
directly sensitive to the initial-state changes, since CAPE feeds straight into updraft
strength — so it is the natural place to look first if a result moves.

In [ ]:
def plot_figure_10_storm_development(arm_label, development_dfs):
    """Render Figure 10 for one arm. Body carried over verbatim from the template."""
    figure_name = f"f10_storm_development_stats__{arm_label}"
    figure_data = {}

    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(10, 8))
    fig.subplots_adjust(hspace=0.6, left=0.17, wspace=0.9, bottom=0.2)

    def ss_max_w(axs, ax_title_start_ix=0):
        updraft_axs = []
        for storm_ix, (storm, label) in enumerate(blt.STORM_LABELS.items()):
            ax = axs[storm_ix]
            development_df = development_dfs[storm]

            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_max_updraft",
                ax.plot,
                development_df["t_minutes"],
                development_df["max_updraft"],
                label="Max updraft W",
                color=cm.plotting.get_nth_color(0),
            )
            updraft_twin_ax = ax.twinx()
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_updraft_mass",
                updraft_twin_ax.plot,
                development_df["t_minutes"],
                development_df["updraft_mass"],
                label="Updraft mass",
                color=cm.plotting.get_nth_color(1),
            )
            updraft_axs.append(updraft_twin_ax)
            # Empty line so the legend picks up the twin axis's series
            ax.plot(
                [], [], color=cm.plotting.get_nth_color(1), label="Updraft mass"
            )
            ax.set_xlabel("Simulation time (minutes)")
            ax.set_ylabel("Updraft speed (m/s)")
            ax.set_title(
                blt.axes_title(
                    ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]
                )
            )
            updraft_twin_ax.set_ylabel("Updraft mass (kg)")
        axs[1].legend(loc="center", bbox_to_anchor=(0.5, -0.35), ncol=2)

    def ss_rain(axs, ax_title_start_ix=0):
        for storm_ix, (storm, label) in enumerate(blt.STORM_LABELS.items()):
            ax = axs[storm_ix]
            development_df = development_dfs[storm]
            for threshold_ix in range(len(blt.RS_TRACER_THRESHOLD_RAIN_RATES)):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rain-{int(blt.RS_TRACER_THRESHOLDS_MM_HR[threshold_ix])}mmhr_area",
                    ax.plot,
                    development_df["t_minutes"],
                    # Gridpoint count -> square kilometres
                    development_df[f"rain_threshold{threshold_ix}"]
                    * 500**2
                    / 1e6,
                    label=(
                        f"{blt.RS_TRACER_THRESHOLDS_MM_HR[threshold_ix]} mm/hr"
                    ),
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                        threshold_ix / len(blt.RS_TRACERS)
                    ),
                )
            ax.set_xlabel("Simulation time (minutes)")
            ax.set_ylabel(r"Area ($km^{2}$)")
            ax.set_title(
                blt.axes_title(
                    ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]
                )
            )
        axs.flatten()[1].legend(
            loc="center", bbox_to_anchor=(0.5, -0.35), ncol=5
        )

    ss_max_w(axs[0])
    ss_rain(axs[1], ax_title_start_ix=3)

    cm.plotting.add_row_header(
        axs[0, 0],
        "Updraft max speed\n" r"and mass $(M)$",
        xycoords="figure fraction",
        xy=(0.14, 0.72),
        fontsize=22,
    )
    cm.plotting.add_row_header(
        axs[1, 0],
        "Rainfall area",
        xycoords="figure fraction",
        xy=(0.14, 0.25),
        fontsize=22,
    )

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(fig, figure_name, FIGURE_OUTPUT_DIR)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        plot_figure_10_storm_development(
            arm_label, development_dfs_by_arm[arm_label]
        )

## 2.4 Figures 11–12 — lofting to upper levels

### 2.4.1 Validate the coarse mass-flux estimator first

The manuscript's upward mass flux through 10 km comes from 2-second lite output. The
harmonized runs have none (`FRQLITE = 0`), so both arms must instead use a 5-minute
estimator. Before relying on it, measure how far it lands from the exact answer on the
**paper** runs, which do have lite output and an already-cached exact flux.

Five-minute sampling aliases the convective-scale vertical velocity field badly, so a large
discrepancy here would not be surprising. What matters is whether the discrepancy is
**similar across the three storms** — a common-mode bias leaves the cross-storm comparison
usable, a storm-dependent one does not.

In [ ]:
UPPER_LEVEL_Z_THRESHOLD_M = blt.UL_Z_THRESHOLD * 1000

# Coarse 5-minute estimator on the paper runs.
coarse_flux_dss_original = {
    storm_key: (
        hu.calculate_cumulative_mass_flux_from_analysis_output(
            storm_ds,
            upper_level_z_threshold_m=UPPER_LEVEL_Z_THRESHOLD_M,
            output_interval_seconds=DELTA_T,
            horizontal_grid_spacing_m=DELTA_X,
        ).compute()
    )
    for storm_key, storm_ds in storm_dss_by_arm[ORIGINAL_ARM_LABEL].items()
}

# The manuscript's exact 2-second flux, already cached from the paper analysis.
exact_flux_cache_path = (
    blt.UL_FLUXES_DIR / f"ul_mass_fluxes_z-{blt.UL_Z_THRESHOLD}000.pkl"
)
print(f"Reading exact (2 s) fluxes from {exact_flux_cache_path}")
exact_flux_dss_original = pd.read_pickle(exact_flux_cache_path)

# Compare the two estimators at the end of the simulation, per storm.
flux_validation_rows = []
for storm_key, coarse_flux_ds in coarse_flux_dss_original.items():
    if storm_key not in exact_flux_dss_original:
        print(f"No exact flux cached for {storm_key}; skipping")
        continue
    exact_flux_ds = exact_flux_dss_original[storm_key]
    coarse_upward = float(
        coarse_flux_ds["cumu_upward_mass_flux"].isel(time=-1).values
    )
    exact_upward = float(
        exact_flux_ds["cumu_upward_mass_flux"].isel(time=-1).values
    )
    flux_validation_rows.append({
        "storm": blt.STORM_LABELS[storm_key],
        "exact_2s_upward_kg": exact_upward,
        "coarse_5min_upward_kg": coarse_upward,
        "coarse_over_exact": coarse_upward / exact_upward,
    })
flux_validation_table = pd.DataFrame(flux_validation_rows).set_index("storm")
display(
    HTML(
        "<h4>Coarse (5 min) vs exact (2 s) upward mass flux, end of"
        " simulation</h4>"
    )
)
display(flux_validation_table)

# The decisive number: how much the coarse/exact ratio itself varies between storms.
# If this is small, the bias is common-mode and the cross-storm comparison survives.
_ratio_spread = flux_validation_table["coarse_over_exact"]
print(
    f"\ncoarse/exact ratio across storms: min={_ratio_spread.min():.4f},"
    f" max={_ratio_spread.max():.4f},"
    f" spread={100 * (_ratio_spread.max() / _ratio_spread.min() - 1):.2f}%"
    " (small => common-mode bias => Figures 11-12 remain comparable between"
    " arms)"
)

In [ ]:
# Coarse estimator for both arms, so the two are computed identically.
coarse_flux_dss_by_arm = {ORIGINAL_ARM_LABEL: coarse_flux_dss_original}
for arm_label in RUN_DIRS_BY_ARM:
    if arm_label in coarse_flux_dss_by_arm:
        continue
    coarse_flux_dss_by_arm[arm_label] = {
        storm_key: (
            hu.calculate_cumulative_mass_flux_from_analysis_output(
                storm_ds,
                upper_level_z_threshold_m=UPPER_LEVEL_Z_THRESHOLD_M,
                output_interval_seconds=DELTA_T,
                horizontal_grid_spacing_m=DELTA_X,
            ).compute()
        )
        for storm_key, storm_ds in tqdm(storm_dss_by_arm[arm_label].items())
    }

In [ ]:
# Upper-level tracer totals for both arms, cached per run so a rerun is cheap.
# Deliberately NOT restricted to the supercell's right mover, matching the manuscript.
upper_level_tracer_dfs_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    arm_upper_level_dfs = {}
    for storm_key, storm_ds in tqdm(storm_dss_by_arm[arm_label].items()):
        cache_dir = hu.upper_level_derived_dir(run_dirs[storm_key], storm_key)
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = (
            cache_dir / f"ul_tracer_totals_z-{UPPER_LEVEL_Z_THRESHOLD_M}.pkl"
        )
        # hu.read_or_cache_dataframe rather than cm.utils.read_or_cache_to: the latter
        # cannot actually write .pkl files and silently no-ops the cache (see that
        # helper's docstring), so every rerun would recompute this from scratch.
        #
        # expected_row_count is what makes a cache written mid-run self-correcting.
        # One row per output time is the shape of this dataframe, so a cache with a
        # different length was written before the run finished and must be redone.
        arm_upper_level_dfs[storm_key] = hu.read_or_cache_dataframe(
            cache_path,
            hu.calculate_upper_level_tracer_dataframe,
            storm_ds,
            UPPER_LEVEL_Z_THRESHOLD_M,
            expected_row_count=storm_ds.sizes["time"],
        )
    upper_level_tracer_dfs_by_arm[arm_label] = arm_upper_level_dfs

In [ ]:
# Precalculate the normalized quantities Figure 11 plots.
for arm_label, arm_upper_level_dfs in upper_level_tracer_dfs_by_arm.items():
    for storm_key, upper_level_df in arm_upper_level_dfs.items():
        cumulative_upward_flux = coarse_flux_dss_by_arm[arm_label][storm_key][
            "cumu_upward_mass_flux"
        ].values

        # Fixed-source tracer aloft, per unit of air lofted through the level
        upper_level_df["ul_fs_tracer_normalized-flux"] = (
            upper_level_df["total_fs_tracer_nc"].values / cumulative_upward_flux
        )

        for tracer in blt.RS_TRACERS:
            upper_level_df[f"ul_{tracer}_nc_normalized-flux"] = (
                upper_level_df[f"{tracer}_nc"].values / cumulative_upward_flux
            )
            # Normalize by the emitted amount from the full storm dataset rather than the
            # precomputed profiles, because the profiles cover only the supercell's right
            # mover while the amount emitted must be for the full domain.
            upper_level_df[f"ul_{tracer}_nc_normalized-acc"] = upper_level_df[
                f"{tracer}_nc"
            ].values / storm_dss_by_arm[arm_label][storm_key][
                f"ACC{tracer}_nc"
            ].isel(
                {"z": 1}
            ).sum(
                ["x", "y"]
            )

In [ ]:
def plot_figure_11_upper_level_tracers(arm_label, upper_level_tracer_dfs):
    """Render Figure 11 for one arm.

    Rows 1 and 2 are normalized by the *coarse* upward mass flux rather than the
    manuscript's 2-second flux, for the reason given at the top of this section.
    Row 3 (normalized by the amount emitted) does not involve the flux at all and is
    therefore directly comparable to the manuscript.
    """
    figure_name = f"f11_tracer_entrainment_{blt.UL_Z_THRESHOLD}km__{arm_label}"
    cm.plotting.start_figure(figure_name, verbose=False)

    bbox_to_anchor = (0.5, -0.4)

    fig, axs = plt.subplots(
        ncols=3, nrows=3, figsize=(8, 12), sharex=True, sharey="row"
    )
    fig.subplots_adjust(bottom=0.4, hspace=0.6)

    for storm_ix, (storm, this_ul_tracer_df) in enumerate(
        upper_level_tracer_dfs.items()
    ):
        # Fixed-source
        axs[0, storm_ix].plot(
            this_ul_tracer_df["t_minutes"],
            this_ul_tracer_df["ul_fs_tracer_normalized-flux"].values,
            name=f"storm-{storm}_tracer-fstracernc_normalized-flux",
        )
        axs[0, storm_ix].set_title(
            blt.axes_title(storm_ix, blt.STORM_LABELS[storm])
        )

        for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
            # Normalized by the air lofted through the level
            axs[1, storm_ix].plot(
                this_ul_tracer_df["t_minutes"],
                this_ul_tracer_df[f"ul_{tracer}_nc_normalized-flux"].values,
                color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                    tracer_ix / len(blt.RS_TRACERS)
                ),
                label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                name=f"storm-{storm}_tracer-rstracer{tracer.lower()}_normalized-flux",
            )
            # Normalized by the amount emitted (flux-independent)
            axs[2, storm_ix].plot(
                this_ul_tracer_df["t_minutes"],
                this_ul_tracer_df[f"ul_{tracer}_nc_normalized-acc"].values,
                color=mpl.colormaps[blt.PROGRESSIVE_CMAP](
                    tracer_ix / len(blt.RS_TRACERS)
                ),
                label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                name=f"storm-{storm}_tracer-rstracer{tracer.lower()}_normalized-acc",
            )

        axs[1, storm_ix].set_title(
            blt.axes_title(3 + storm_ix, blt.STORM_LABELS[storm])
        )
        axs[2, storm_ix].set_title(
            blt.axes_title(6 + storm_ix, blt.STORM_LABELS[storm])
        )

    cm.plotting.add_row_header(
        axs[0, 0], r"$N_{FS-UL}$/" + "\n" + r"$F_{upward}$", pad=80, fontsize=22
    )
    cm.plotting.add_row_header(
        axs[1, 0], r"$N_{RS-UL}$/" + "\n" + r"$F_{upward}$", pad=80, fontsize=22
    )
    cm.plotting.add_row_header(
        axs[2, 0], r"$N_{RS-UL}$/" + "\n" + r"$E_{RS}$", pad=80, fontsize=22
    )

    axs[0, 0].set_ylabel("#/kg")
    axs[0, 0].ticklabel_format(axis="y", style="sci", scilimits=(4, 4))
    axs[1, 0].set_ylabel("#/kg")
    axs[2, 0].set_ylabel("Fraction")

    for ax in axs.flatten():
        blt.label_x_axis(ax)
    for ax in axs[1:2].flatten():
        ax.set_yscale("log")
        ax.set_ylim([1e0, 1e5])

    axs[1, 1].legend(
        loc="center",
        bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1] + 0.1),
        ncol=5,
    )
    axs[2, 1].legend(
        loc="center",
        bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1]),
        ncol=5,
    )

    fig.suptitle(
        f"[{arm_label}] rows 1-2 use the coarse 5-min flux",
        fontsize=12,
        y=0.995,
    )

    cm.plotting.save_figure_and_data(
        figure_output_dirs=[FIGURE_OUTPUT_DIR],
        data_output_dirs=[FIGURE_DATA_OUTPUT_DIR],
        extensions=[".pdf", ".png"],
    )
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        plot_figure_11_upper_level_tracers(
            arm_label, upper_level_tracer_dfs_by_arm[arm_label]
        )

In [ ]:
def plot_figure_12_upper_level_fluxes(arm_label, coarse_flux_dss):
    """Render Figure 12 (cumulative up/down mass flux through 10 km) for one arm.

    Both curves use the coarse 5-minute estimator, so the absolute values will not match
    the manuscript's. Their *ratio* (below) is the quantity the manuscript actually quotes
    and is much less sensitive to the sampling interval.
    """
    fig, axs = plt.subplots(
        ncols=3, figsize=(9, 3.2), layout="constrained", sharex=True
    )
    for storm_ix, (storm, this_ul_flux_ds) in enumerate(
        coarse_flux_dss.items()
    ):
        axs[storm_ix].plot(
            this_ul_flux_ds["t_minutes"],
            this_ul_flux_ds["cumu_upward_mass_flux"],
            label="Upward",
        )
        axs[storm_ix].plot(
            this_ul_flux_ds["t_minutes"],
            -this_ul_flux_ds["cumu_downward_mass_flux"],
            label="Downward",
        )
        axs[storm_ix].set_title(
            blt.axes_title(storm_ix, blt.STORM_LABELS[storm])
        )
        blt.label_x_axis(axs[storm_ix])
        axs[storm_ix].set_ylabel("kg")
    cm.plotting.clean_legend(axs[0], loc="upper left")
    fig.suptitle(
        f"[{arm_label}] cumulative mass flux through {blt.UL_Z_THRESHOLD} km"
        " (coarse 5-min)"
    )
    cm.plotting.fig_multisave(
        fig,
        f"f12_ul_fluxes_{blt.UL_Z_THRESHOLD}km__{arm_label}",
        FIGURE_OUTPUT_DIR,
    )
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        plot_figure_12_upper_level_fluxes(
            arm_label, coarse_flux_dss_by_arm[arm_label]
        )

In [ ]:
# Net flux ratio at the end of the simulation: downward over upward. A ratio, so the
# coarse-sampling bias largely cancels.
net_flux_ratio_rows = []
for arm_label, coarse_flux_dss in coarse_flux_dss_by_arm.items():
    for storm_key, this_flux_ds in coarse_flux_dss.items():
        net_flux_ratio_rows.append({
            "arm": arm_label,
            "storm": blt.STORM_LABELS[storm_key],
            "downward_over_upward": float(
                (
                    this_flux_ds["cumu_downward_mass_flux"]
                    / -this_flux_ds["cumu_upward_mass_flux"]
                )
                .isel({"time": -1})
                .values
            ),
        })
display(HTML("<h4>Net flux ratio at end of simulation</h4>"))
display(pd.DataFrame(net_flux_ratio_rows).set_index(["arm", "storm"]).round(4))

## 2.5 Tracer entrainment statistics

The full statistics table from the manuscript notebook, per arm, followed by the
storm-to-storm ratio tables. The ratio tables are the ones to read: they are the numbers the
manuscript's comparative claims rest on.

In [ ]:
def calculate_tracer_statistics(tracer_totals_dss):
    """Compute the manuscript's full tracer entrainment statistics for one arm.

    Body carried over verbatim from `validation_comparison.ipynb`; only the data source
    is now an argument rather than a stashed global.
    """
    tracer_stats = {}
    tracer_stats_tracers = blt.RS_TRACERS + ["total_fs_tracer"]

    for storm_ix, (storm, this_tracer_ds) in enumerate(
        tracer_totals_dss.items()
    ):
        # Limit to times when an updraft actually exists, or the per-kg means divide by zero
        this_tracer_ds = this_tracer_ds.where(this_tracer_ds["air_mass"] > 0)
        this_storm_stats = {}
        for ix, tracer in enumerate(tracer_stats_tracers):
            mean_mixing_ratio = (
                this_tracer_ds[tracer + "_nc"] / this_tracer_ds["air_mass"]
            )
            entrainment_efficiency = (
                this_tracer_ds[tracer + "_nc"]
                / this_tracer_ds["ACC" + tracer + "_nc"]
            )
            rain_production_efficiency = (
                this_tracer_ds["ACC" + tracer + "_nc"]
                / this_tracer_ds["air_mass"]
            )
            tracer_fraction = (
                this_tracer_ds[tracer + "_nc"]
                / this_tracer_ds["total_fs_tracer_nc"]
            )
            # Output is 5-minutely, so index 20 is t=100 min and index 36 is t=180 min
            this_post_100_ds = this_tracer_ds.isel({"time": slice(20, 37)})
            post_100_mixing_ratio = (
                this_post_100_ds[tracer + "_nc"] / this_post_100_ds["air_mass"]
            )

            this_storm_stats[tracer] = {
                "mean_mean": mean_mixing_ratio.mean(...).values,
                "max_mean": mean_mixing_ratio.max(...).values,
                "max_mean_time": (
                    this_tracer_ds["t_minutes"].values[
                        mean_mixing_ratio.argmax("time").values
                    ]
                ),
                "eos_mean": mean_mixing_ratio.isel({"time": -1}).values,
                "max_mean_post_t100": post_100_mixing_ratio.max(...).values,
                "min_mean_post_t100": post_100_mixing_ratio.min(...).values,
                "mean_tracer_fraction": tracer_fraction.mean().values,
                "max_tracer_fraction": tracer_fraction.max().values,
                "max_tracer_fraction_time": (
                    this_tracer_ds["t_minutes"].values[
                        tracer_fraction.argmax("time").values
                    ]
                ),
                "mean_rain_production_efficiency": (
                    rain_production_efficiency.mean().values
                ),
                "max_rain_production_efficiency": (
                    rain_production_efficiency.max().values
                ),
                "eos_rain_production_efficiency": (
                    rain_production_efficiency.isel({"time": -1}).values
                ),
                "integrated_rain_production_efficiency": (
                    rain_production_efficiency.sum().values
                ),
                "t90_rain_production_efficiency": (
                    rain_production_efficiency.isel({"time": 90 // 5}).values
                ),
                "t100_rain_production_efficiency": (
                    rain_production_efficiency.isel({"time": 100 // 5}).values
                ),
                "t180_over_t75_rain_production_efficiency": (
                    (
                        rain_production_efficiency.isel({"time": 180 // 5})
                        / rain_production_efficiency.isel({"time": 75 // 5})
                    ).values
                ),
                "t180_over_t75_updraft_mass": (
                    (
                        this_tracer_ds["air_mass"].isel({"time": 180 // 5})
                        / this_tracer_ds["air_mass"].isel({"time": 75 // 5})
                    ).values
                ),
                "max_entrainment_efficiency": (
                    entrainment_efficiency.max().values
                ),
                "max_entrainment_efficiency_time": (
                    this_tracer_ds["t_minutes"].values[
                        entrainment_efficiency.argmax("time").values
                    ]
                ),
                "mean_entrainment_efficiency": (
                    entrainment_efficiency.mean().values
                ),
                "integrated_entrainment_efficiency": (
                    entrainment_efficiency.sum().values
                ),
                "eos_entrainment_efficiency": (
                    entrainment_efficiency.isel({"time": -1}).values
                ),
                "t120_to_t180_mean_entrainment_efficiency": (
                    entrainment_efficiency.isel({"time": slice(24, 37)})
                    .mean()
                    .values
                ),
                "t85_to_t180_mean_entrainment_efficiency": (
                    entrainment_efficiency.isel({"time": slice(17, 37)})
                    .mean()
                    .values
                ),
                "t90_entrainment efficiency": (
                    entrainment_efficiency.isel({"time": 90 // 5}).values
                ),
            }
        tracer_stats[storm] = this_storm_stats
    return tracer_stats


tracer_stats_by_arm = {}
for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        tracer_stats_by_arm[arm_label] = calculate_tracer_statistics(
            tracer_totals_dss_by_arm[arm_label]
        )
        blt.save_figure_data(
            tracer_stats_by_arm[arm_label],
            f"entrainment_factor_stats__{arm_label}",
        )

    display(HTML(f"<h2>{arm_label}</h2>"))
    for storm_key, storm_stats in tracer_stats_by_arm[arm_label].items():
        display(HTML(f"<h3>{blt.STORM_LABELS[storm_key]}</h3>"))
        display(pd.DataFrame(storm_stats))

In [ ]:
# Storm-to-storm ratios of the full statistics table, side by side across arms.
# A ratio that is stable between arms means the comparative claim it supports is robust
# to the setup differences.
for comparison_pair in [
    ["sl_wk", "ic_wk"],
    ["sc_wk", "ic_wk"],
    ["sc_wk", "sl_wk"],
    ["sl_wk", "sc_wk"],
    ["ic_wk", "sl_wk"],
    ["ic_wk", "sc_wk"],
]:
    numerator_storm, denominator_storm = comparison_pair
    display(
        HTML(
            f"<h2>{blt.STORM_LABELS[numerator_storm]} /"
            f" {blt.STORM_LABELS[denominator_storm]}</h2>"
        )
    )
    for arm_label, tracer_stats in tracer_stats_by_arm.items():
        display(HTML(f"<h4>{arm_label}</h4>"))
        display(
            pd.DataFrame(tracer_stats[numerator_storm])
            / pd.DataFrame(tracer_stats[denominator_storm])
        )

## 2.6 Cross-storm summary — the fairness test

This is what the notebook exists to produce.

The first table shows whether each storm's own numbers moved. That is interesting but not
decisive: a setup error that shifts all three storms by the same factor leaves every
comparative claim in the manuscript intact, because every one of them is a comparison.

The second table is the decisive one. It forms the ratio of each headline metric between
each pair of storms, in each arm, and reports how far that ratio moves. **A ratio that
survives harmonization means the cross-storm claim resting on it is robust to the setup
differences. A ratio that changes materially — or flips — means it is not.**

Rows whose change exceeds 10% are highlighted.

In [ ]:
# Reduce each arm to a small set of per-storm headline numbers.
headline_metrics_by_arm = {}
for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(
        RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR
    ):
        headline_metrics_by_arm[arm_label] = hu.collect_headline_metrics(
            tracer_totals_dss_by_arm[arm_label],
            upper_level_tracer_dfs=upper_level_tracer_dfs_by_arm[arm_label],
            development_dfs=development_dfs_by_arm[arm_label],
        )
    display(HTML(f"<h4>Headline metrics — {arm_label}</h4>"))
    display(headline_metrics_by_arm[arm_label])

In [ ]:
# Did each storm's own numbers move?
display(HTML("<h3>Per storm: original vs harmonized</h3>"))
per_storm_comparison = hu.compare_arms_table(
    headline_metrics_by_arm, ORIGINAL_ARM_LABEL, HARMONIZED_ARM_LABEL
)
display(
    hu.style_percent_change(
        per_storm_comparison, "percent_change", threshold=10.0
    )
)

In [ ]:
# THE fairness test: did the ratios *between* storms move?
display(
    HTML(
        "<h3>Cross-storm ratios: does the comparison between storms"
        " survive?</h3>"
    )
)
cross_storm_ratios = hu.cross_storm_ratio_table(headline_metrics_by_arm)
display(
    hu.style_percent_change(
        cross_storm_ratios, "ratio_percent_change", threshold=10.0
    )
)

# Compact read: the largest ratio movement per storm pair, which is the worst case for
# any comparative claim involving that pair.
display(HTML("<h4>Worst-case ratio movement per storm pair</h4>"))
display(
    cross_storm_ratios["ratio_percent_change"]
    .abs()
    .groupby("storm_pair")
    .agg(["max", "mean"])
    .round(2)
)

## 2.7 Sanity check: does the supercell right-mover mask still fit?

The supercell's analysis is restricted to its right mover by a **fixed geometric line**
(`blt.sc_wk_right_mover_boundary`, `y ≤ 78000 + x/5`), chosen by eye against the manuscript
run. A rerun supercell can split at a different time or place, in which case that fixed line
would clip the wrong part of the storm — and it would do so silently, because nothing in the
pipeline checks it.

This overlays the boundary on each arm's supercell condensate field at several times. The
right mover should sit below the line in both arms.

In [ ]:
# Overlay the right-mover boundary on the supercell's column-maximum condensate.
SNAPSHOT_TIME_INDICES = [
    12,
    20,
    28,
    36,
]  # 5-minutely output, so t = 60, 100, 140, 180 min

fig, axs = plt.subplots(
    nrows=len(RUN_DIRS_BY_ARM),
    ncols=len(SNAPSHOT_TIME_INDICES),
    figsize=(4 * len(SNAPSHOT_TIME_INDICES), 3.6 * len(RUN_DIRS_BY_ARM)),
    layout="constrained",
    squeeze=False,
)

for arm_ix, arm_label in enumerate(RUN_DIRS_BY_ARM):
    supercell_ds = storm_dss_by_arm[arm_label]["sc_wk"]
    for time_ix, snapshot_time_index in enumerate(SNAPSHOT_TIME_INDICES):
        ax = axs[arm_ix, time_ix]
        # Column-maximum condensate makes the storm's footprint visible in one panel
        condensate_field = (
            supercell_ds["R_condensate"]
            .isel({"time": snapshot_time_index})
            .max("z")
            .compute()
        )
        ax.pcolormesh(
            supercell_ds["x"].values / 1000,
            supercell_ds["y"].values / 1000,
            condensate_field.transpose("y", "x").values,
            shading="auto",
        )
        # The fixed boundary the analysis mask uses
        boundary_x = supercell_ds["x"].values
        ax.plot(
            boundary_x / 1000,
            blt.sc_wk_right_mover_boundary(boundary_x) / 1000,
            color="red",
            linewidth=2,
            label="right-mover boundary",
        )
        ax.set_title(
            f"{arm_label}\nt ="
            f" {int(supercell_ds['t_minutes'].isel(time=snapshot_time_index))} min",
            fontsize=11,
        )
        ax.set_xlabel("x (km)")
        if time_ix == 0:
            ax.set_ylabel("y (km)")

axs[0, 0].legend(loc="upper left", fontsize=9)
fig.suptitle(
    "Supercell column-max condensate with the fixed right-mover boundary"
)
cm.plotting.fig_multisave(
    fig, "p2_supercell_right_mover_check", FIGURE_OUTPUT_DIR
)

## 2.8 Storm animations

An animation of each storm's full evolution, with the two arms stacked so the same quantity
can be read straight down a column at the same model time.

The four columns span the storm from the ground up:

1. **Surface rain rate**, with the cold-pool boundary outlined (the −0.04 m/s² low-level
   buoyancy contour used in the manuscript's Figures 3–5) and a gold line marking where the
   cross section is taken. Rain rate is what seeds the rain-sourced tracer, so this panel is
   the origin of everything the study measures.
2. **Column-maximum condensate** with 5 km vertical-velocity contours — the storm's footprint
   and where it is actively lifting.
3. **Storm-following cross section of condensate** with updraft/downdraft contours — the
   vertical structure, including whether an anvil forms and how deep the storm gets.
4. **The same cross section in buoyancy** — the cold pool, which drives the low-level
   convergence that either sustains the updraft or undercuts it.

Two choices matter for reading these correctly:

- **Colour limits are fixed** across every frame and shared between arms. If each frame
  autoscaled, a storm that weakened over time would look constant; if each arm autoscaled
  separately, the two rows would be visually incomparable and the figure would be worse than
  useless for the comparison it exists to make.
- **Cross sections follow the storm**, taken at each frame's storm centroid rather than at a
  fixed y. A fixed cross section drifts off a moving storm and shows empty air for exactly the
  later frames where the arms differ most.

In [ ]:
# Animation parameters. The strides trade detail for time and file size; stride 2 in space
# quarters the cell count for a barely visible loss, and is what keeps the GIFs manageable.
ANIMATION_SPATIAL_STRIDE = 2  # every Nth gridpoint in x and y
ANIMATION_FRAME_STRIDE = 1  # every Nth output time (1 = all 37)
ANIMATION_FPS = 4
EMBED_ANIMATIONS_INLINE = (
    True  # set False to keep the notebook small (GIFs still written)
)

ANIMATION_DIR = OUTPUT_DIR / "animations"
ANIMATION_DIR.mkdir(parents=True, exist_ok=True)

# Reduce the 4-D output to the six 2-D fields the animation needs, one dask pass per run.
#
# Deliberately NOT cached to disk: the reduced fields run to roughly 95 MB per storm-arm,
# which would put ~570 MB into the repo's derived tree to save about five minutes of
# recomputation. Not a trade worth making, and it removes a class of stale-cache bugs.
animation_fields_by_storm_and_arm = {}

for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    animation_fields_by_storm_and_arm[storm_key] = {}
    for arm_label in RUN_DIRS_BY_ARM:
        print(f"Computing animation fields for {storm_key} [{arm_label}]...")
        animation_fields_by_storm_and_arm[storm_key][arm_label] = (
            hu.precompute_storm_animation_fields(
                storm_dss_by_arm[arm_label][storm_key],
                spatial_stride=ANIMATION_SPATIAL_STRIDE,
                frame_stride=ANIMATION_FRAME_STRIDE,
            )
        )

print(
    "\nDone. Frames per animation:",
    {
        k: v[ORIGINAL_ARM_LABEL].sizes["time"]
        for k, v in animation_fields_by_storm_and_arm.items()
    },
)

### Storm structure in one still figure

The animations show everything, which is more than a reader can hold at once. This figure
keeps only the part that carries the arm-to-arm difference, so that the comparison can be
made without playing anything back.

Only the **storm-following vertical cross section** is kept. Every consequence of a changed
sounding shows up in it at once: how deep the updraft reaches, how much condensate makes it
into the anvil, how far the storm has propagated, and how strong the cold pool underneath has
grown. The plan views are dropped.

Rows are the two arms and columns are late model times. Late, because both arms start from
the same trigger and only separate as the storms mature. Reading straight down a column
compares the arms at the same model minute on identical colour limits, so a visible
difference is a real difference rather than an autoscaling artefact.

Two things to be aware of when reading these:

- Panels are in **absolute domain coordinates**, sharing one x window per column, rather than
  being re-centred on each arm's own storm. A difference in propagation speed is one of the
  effects being looked for here, and re-centring each panel would subtract exactly that
  effect out.
- The cold-pool contour is **cut off above 4 km**. The −0.04 m/s² threshold is also met by
  the negatively buoyant air around the anvil, and contouring that scatters closed loops
  across the top of every panel.

`plot_storm_structure_comparison` also takes `include_overlay_row=True`, which adds a third
row that shades neither arm and instead draws both arms' outlines — condensate edge, updraft
core and cold pool — one colour per arm. It turns a difference in storm depth into a distance
on the page rather than something estimated from memory. It is off here because it costs a
third of the figure's height and the two shaded rows already carry the comparison.

In [ ]:
# The still-figure summary of each storm, from the same precomputed fields the animations use.
#
# Panels are deliberately small. Every text size in the figure is a multiple of the active
# `font.size`, so shrinking the panels without touching the rcParams is what makes the
# annotations large relative to the storms. Set `include_overlay_row=True` for the third,
# outlines-only row described above.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_storm_structure_comparison(
        animation_fields_by_storm_and_arm[storm_key],
        storm_label=blt.STORM_LABELS[storm_key],
        panel_width=1.5,
        panel_height=1.2,
    )
    cm.plotting.fig_multisave(
        fig, f"p2_storm_structure_comparison_{storm_key}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# Build one animation per storm, each with both arms stacked. Rendering redraws every panel
# per frame (filled contours cannot be updated in place), so this takes a couple of minutes
# per storm.
from IPython.display import Image

animation_gif_paths = {}

for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    gif_path = ANIMATION_DIR / f"storm_animation_{storm_key}.gif"
    hu.animate_storm_comparison(
        animation_fields_by_storm_and_arm[storm_key],
        storm_label=blt.STORM_LABELS[storm_key],
        output_gif_path=gif_path,
        frames_per_second=ANIMATION_FPS,
    )
    animation_gif_paths[storm_key] = gif_path

In [ ]:
# Build one animation per storm, each with both arms stacked. Rendering redraws every panel
# per frame (filled contours cannot be updated in place), so this takes a couple of minutes
# per storm.
from IPython.display import Image

animation_gif_paths = {}

for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    gif_path = ANIMATION_DIR / f"storm_animation_pv_{storm_key}.gif"
    hu.animate_storm_comparison(
        animation_fields_by_storm_and_arm[storm_key],
        storm_label=blt.STORM_LABELS[storm_key],
        output_gif_path=gif_path,
        frames_per_second=ANIMATION_FPS,
    )
    animation_gif_paths[storm_key] = gif_path

In [ ]:
# Show the animations inline. Each GIF is embedded in the notebook, so this is what makes the
# notebook self-contained -- and also what makes it large. Set EMBED_ANIMATIONS_INLINE = False
# above to keep the notebook small; the GIFs are written to disk either way.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    gif_path = animation_gif_paths[storm_key]
    display(HTML(f"<h3>{blt.STORM_LABELS[storm_key]}</h3>"))
    if EMBED_ANIMATIONS_INLINE:
        display(Image(filename=str(gif_path)))
    else:
        print(f"  {gif_path}  ({gif_path.stat().st_size / 1e6:.1f} MB)")

# New figures

In [ ]:
from importlib import reload

reload(hu)

# Reduce each run to the five 2-D fields this animation needs, one dask pass per run.
#
# Deliberately not cached to disk, for the same reason as the §2.8 fields: the reduction runs
# to tens of MB per storm-arm, which is not worth putting in the repo to save a few minutes.
w_condensate_rainfall_fields_by_storm_and_arm = {}

for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    w_condensate_rainfall_fields_by_storm_and_arm[storm_key] = {}
    for arm_label in RUN_DIRS_BY_ARM:
        print(
            f"Computing W/condensate/rainfall fields for {storm_key}"
            f" [{arm_label}]..."
        )
        w_condensate_rainfall_fields_by_storm_and_arm[storm_key][arm_label] = (
            hu.precompute_w_condensate_rainfall_fields(
                storm_dss_by_arm[arm_label][storm_key],
                spatial_stride=ANIMATION_SPATIAL_STRIDE,
                frame_stride=ANIMATION_FRAME_STRIDE,
            )
        )

# Report what actually went into the accumulation, so a caption cannot misstate it.
_any_wcr_fields = w_condensate_rainfall_fields_by_storm_and_arm[
    hu.STORM_KEYS_IN_FIGURE_ORDER[0]
][ORIGINAL_ARM_LABEL]
print(
    "\nAccumulated precipitation sums:",
    _any_wcr_fields.attrs["accumulated_precipitation_variables"],
)
print("W plan view taken at:", _any_wcr_fields.attrs["plan_view_height_m"], "m")

In [ ]:
# One animation per storm, both arms stacked. Five columns of redrawn axes per frame, so
# this takes a few minutes per storm.
w_condensate_rainfall_gif_paths = {}

for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    gif_path = ANIMATION_DIR / f"w_condensate_rainfall_{storm_key}.gif"
    hu.animate_w_condensate_rainfall(
        w_condensate_rainfall_fields_by_storm_and_arm[storm_key],
        storm_label=blt.STORM_LABELS[storm_key],
        output_gif_path=gif_path,
        frames_per_second=ANIMATION_FPS,
    )
    w_condensate_rainfall_gif_paths[storm_key] = gif_path

In [ ]:
from IPython.display import Image

# Show them inline. As in §2.8, embedding is what makes the notebook self-contained and also
# what makes it large; EMBED_ANIMATIONS_INLINE controls it and the GIFs are written either way.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    gif_path = w_condensate_rainfall_gif_paths[storm_key]
    display(HTML(f"<h3>{blt.STORM_LABELS[storm_key]}</h3>"))
    if EMBED_ANIMATIONS_INLINE:
        display(Image(filename=str(gif_path)))
    else:
        print(f"  {gif_path}  ({gif_path.stat().st_size / 1e6:.1f} MB)")

## 2.9 Why the squall line's rain-sourced entrainment fell

§2.5 and §2.6 record that the squall line entrains much less rain-sourced tracer after
harmonization. Neither section says *why*, and the two candidate explanations carry very
different implications for the manuscript.

* **A supply problem.** The harmonized squall line has less CAPE, so it rains less, so there is
  less rain-sourced tracer in existence to entrain. The entrainment machinery is unchanged; it
  simply has less to work with.
* **A transport problem.** The rain-sourced tracer is emitted at the surface and has to be
  drawn into the updraft and carried up it. If that transport weakens, entrainment falls even
  at constant supply.

The two separate cleanly, because the headline metric factors exactly. Write $N$ for the
rain-sourced tracer particles inside the updraft, $A$ for everything the surface rain has ever
emitted, and $M$ for the updraft's air mass. Then

$$ q_{RS} \;\equiv\; \frac{N}{M} \;=\; \underbrace{\frac{A}{M}}_{\eta_{RP}} \;\times\; \underbrace{\frac{N}{A}}_{\eta_{RSE}} $$

so that $\Delta \ln q_{RS} = \Delta \ln A - \Delta \ln M + \Delta \ln \eta_{RSE}$, and the three
contributions have to add up to the observed change. Supply is $A$. Transport is $\eta_{RSE}$.

Two controls run through the whole section.

1. **The fixed-source tracer.** Its 36 components are released from the boundary layer on a
   fixed schedule that takes no account of where it is raining, so it measures how much
   boundary-layer air of *any* origin the updraft is carrying. If the rain-sourced and
   fixed-source tracers fall together, then nothing about the change is specific to rain-cooled
   air.
2. **The other two storms.** The supercell is the same run in both arms of this notebook, so
   every ratio below has to come out at exactly 1.000 for it. That is the arithmetic check on
   the section.

In [ ]:
# Reload the helper module explicitly. `%autoreload 2` picks up edited function bodies, but it
# does not reliably re-bind module-level constants added after the first import, and this
# section adds several.
from importlib import reload

reload(hu)

# The storm this section is about, and the rain-sourced tracer it leads with. TRACERP003 is
# emitted wherever the surface rain rate exceeds 10 mm/hr, which is the threshold
# `collect_headline_metrics` quotes its single numbers from.
ENTRAINMENT_STORM_KEY = "sl_wk"
ENTRAINMENT_RS_TRACER = hu.REPRESENTATIVE_RAIN_SOURCED_TRACER

# A mature-storm window, used alongside the full simulation throughout. The two arms differ in
# *when* their squall lines get going as well as in how much they entrain, and one number over
# the whole run cannot tell those two apart. Anything that survives this window is not a
# timing artefact.
MATURE_STORM_WINDOW_MINUTES = (100.0, np.inf)

print(f"Storm:  {blt.STORM_LABELS[ENTRAINMENT_STORM_KEY]}")
print(
    f"Tracer: {ENTRAINMENT_RS_TRACER}"
    f" (emitted above {hu.rain_sourced_threshold_label(ENTRAINMENT_RS_TRACER)}"
    " of surface rain)"
)
print(f"Control tracer: {hu.BOUNDARY_LAYER_CONTROL_TRACER}")

### 2.9.1 The factor decomposition

Each table below sums $N$, $A$ and $M$ over the window and *then* divides. Averaging the
per-time ratios instead would break the identity, because the mean of $N/A$ is not
$(\sum N)/(\sum A)$, and the contributions would stop adding up. The helper asserts that the
decomposition closes rather than trusting that it does.

In [ ]:
# The decomposition, over the full simulation and then over the mature storm alone.
for window_label, analysis_window_minutes in [
    ("full simulation", (0.0, np.inf)),
    (
        f"t >= {MATURE_STORM_WINDOW_MINUTES[0]:g} min",
        MATURE_STORM_WINDOW_MINUTES,
    ),
]:
    display(
        HTML(
            f"<h4>{blt.STORM_LABELS[ENTRAINMENT_STORM_KEY]} &mdash;"
            f" {window_label}</h4>"
        )
    )
    display(
        hu.rain_sourced_entrainment_factor_table(
            tracer_totals_dss_by_arm,
            ENTRAINMENT_STORM_KEY,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
            analysis_window_minutes=analysis_window_minutes,
        )
    )

# The same decomposition for all three storms, reduced to the ratio column. The supercell is
# the same run in both arms, so its column is the arithmetic check on everything above: every
# entry in it has to be 1.000.
display(HTML("<h4>All storms, ratio column (full simulation)</h4>"))
display(
    pd.DataFrame({
        blt.STORM_LABELS[storm_key]: hu.rain_sourced_entrainment_factor_table(
            tracer_totals_dss_by_arm,
            storm_key,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
        )["ratio"]
        for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER
    }).round(3)
)

In [ ]:
# Every factor against time, both arms. The table cannot distinguish "never got there" from
# "got there late", and those have different implications, so the time series has to be looked
# at directly. Panel (f) carries the decomposition itself.
#
# All three storms are plotted, not just the squall line: a factor that moves the same way in
# every storm is a property of the harmonization rather than of the squall line.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_rain_sourced_entrainment_factors(
        tracer_totals_dss_by_arm,
        storm_key,
        ORIGINAL_ARM_LABEL,
        rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
    )
    cm.plotting.fig_multisave(
        fig, f"rs_entrainment_factors__{storm_key}", FIGURE_OUTPUT_DIR
    )

### 2.9.2 Where in the column the entrainment was lost

A column total cannot tell an *ingestion* failure from a *lofting* failure. A storm that never
gets rain-cooled air into its updraft base, and a storm that gets it in and then fails to carry
it upward, produce the same drop in column-integrated entrainment. They are different storms
with different causes.

Resolving the same counts by height separates them at once. Agreement at low levels with
divergence aloft means the updraft base is still ingesting rain-sourced air, and the loss is in
the lofting. Divergence at the surface means the ingestion itself changed.

The fixed-source tracer is what makes the answer specific. The last panel of the figure divides
the rain-sourced profile ratio by the fixed-source profile ratio, so the residual can be read
straight off: a value of one there means the entire rain-sourced change is explained by the
general boundary-layer change, and nothing about it is specific to rain.

In [ ]:
# Restricted to the mature storm. The early frames contribute almost nothing to the integral,
# and including them only widens the height range that has any data in it at all.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_rain_sourced_entrainment_profiles(
        tracer_totals_profile_dss_by_arm,
        storm_key,
        ORIGINAL_ARM_LABEL,
        rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
        analysis_window_minutes=MATURE_STORM_WINDOW_MINUTES,
    )
    cm.plotting.fig_multisave(
        fig, f"rs_entrainment_profiles__{storm_key}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# The same split expressed as numbers, so it can be quoted.
#
# Read the three pairs of rows against each other. If the tracer centre of gravity collapses
# while the updraft mass centre of gravity barely moves, the storm still has its depth and is
# simply not carrying low-level air up it.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    display(HTML(f"<h4>{blt.STORM_LABELS[storm_key]}</h4>"))
    display(
        hu.rain_sourced_entrainment_height_partition_table(
            tracer_totals_profile_dss_by_arm,
            storm_key,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
            analysis_window_minutes=MATURE_STORM_WINDOW_MINUTES,
        ).T.round(3)
    )

### 2.9.3 The supply side: what happened to the rain

The five rain-sourced tracers are emitted under progressively heavier rain — 1, 5, 10, 20 and
40 mm/hr — so sweeping across them tests *which* property of the rain changed.

* A roughly uniform drop in the emitted count at every threshold means the rain covers less
  ground but is no less intense, which points at storm size.
* A drop that steepens with threshold means the rain is less intense, which points at updraft
  strength and depth, because heavy surface rain needs a deep vigorous updraft to make the
  condensate in the first place.

The entrainment efficiency column is read the other way round. It describes transport of tracer
that already exists, so if it moves by about the same factor at every threshold, then the
transport change belongs to the updraft rather than to the rain.

In [ ]:
# The decomposition repeated at all five rain thresholds at once.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    display(HTML(f"<h4>{blt.STORM_LABELS[storm_key]}</h4>"))
    display(
        hu.rain_threshold_sensitivity_table(
            tracer_totals_dss_by_arm, storm_key, ORIGINAL_ARM_LABEL
        ).round(3)
    )

In [ ]:
# The rain field itself, straight from the development dataframes. This reaches the same
# conclusion as the emitted-tracer column above by a route that never touches the tracers,
# which is what makes it worth drawing separately.
#
# The last panel is the control on the other three: it says whether a storm making less heavy
# rain is also running weaker updraft cores, or only fewer of them.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_rain_area_by_threshold(development_dfs_by_arm, storm_key)
    cm.plotting.fig_multisave(
        fig, f"rain_area_by_threshold__{storm_key}", FIGURE_OUTPUT_DIR
    )

### 2.9.4 What this section establishes

The numbers quoted here come from the run of this notebook that produced the cells above. They
are written down so the argument can be checked against the output, not so they can be copied
forward. Re-run the notebook and re-read them.

**The loss is transport, not supply.** Over the full simulation the squall line's rain-sourced
supply $A$ fell 31%, its updraft mass $M$ fell 53%, and its entrainment efficiency
$\eta_{RSE}$ fell 63%. Because $M$ sits in the denominator, the shrinking updraft *raises*
$\eta_{RP}$ by 48% and so partly conceals the loss: the mixing ratio $q_{RS}$ falls 45% while
the total entrained count falls 74%. Restricting to $t \ge 100$ min changes almost nothing, so
this is not an artefact of the harmonized squall line getting going later.

**The rain-sourced air gets in, but it does not go up.** Below roughly 600 m the two arms
entrain about the same amount. The ratio falls to 0.12 by 2.3 km and stays near 0.1 through the
rest of the column. The count-weighted mean height of entrained rain-sourced tracer drops from
3430 m to 1510 m, and the fraction of it below 2 km rises from 0.47 to 0.80. Over the same
window the updraft mass mean height moves only from 5980 m to 5410 m. The storm keeps its
depth and stops carrying low-level air up it.

**Almost none of this is specific to rain-cooled air.** The fixed-source boundary-layer tracer
falls by the same factor at the same heights. The rain-specific residual sits between 0.85 and
0.90 through the whole column, so roughly 85% of the rain-sourced decline is a general
boundary-layer lofting decline that the rain-sourced tracer inherits.

**The rain that does fall is much less intense.** The emitted count falls monotonically with
threshold: 0.84 at 1 mm/hr, 0.78 at 5, 0.69 at 10, 0.49 at 20 and 0.12 at 40. The area raining
harder than 40 mm/hr nearly disappears. Peak updraft speed is unchanged or slightly higher, so
the harmonized squall line is not running weaker cores. It is running far fewer of them, which
is the same statement as the 53% drop in updraft mass.

**The efficiency loss is not threshold-selective.** The entrainment efficiency ratio sits
between 0.33 and 0.48 at all five thresholds. That is a far narrower spread than the emitted
count's 0.84 down to 0.12, and it does not fall steadily with threshold the way the emitted
count does. Whatever changed acts on transport of tracer that already exists, and it acts on
all of it to about the same degree.

Putting those together: the squall line loses 320 J/kg of surface-based CAPE under this
harmonization (§1.7). That gives it a smaller updraft, which produces much less heavy rain and,
more importantly, lofts far less boundary-layer air above about 1 km. The rain-sourced tracer
decline is mostly the second of those two.

**What it means for the manuscript.** The mechanism is not rain-specific, so the manuscript's
rain-sourced entrainment argument is not resting on an artefact of how the rain-sourced tracers
are defined. It does mean the squall line's *absolute* entrainment numbers move a long way with
the setup, which is the case for continuing to quote the cross-storm ratios of §2.6 rather than
the per-storm values.

## 2.10 Why the isolated convection's entrainment efficiency rose

The isolated convection moves the opposite way from the squall line. It rains over 1.65 times
more ground, and its rain-sourced entrainment efficiency rises by a factor of 2.3. The rain is
expected: this storm gains 740 J/kg of surface-based CAPE (§1.7). The efficiency is not, and
this section is about that half.

**Read the framing first.** This harmonization does something different to the isolated
convection than it does to the squall line. The squall line's changes remove *errors*. The
isolated convection was designed to be drier than the other two storms, at 11 g/kg against
13 g/kg, and putting it on the supercell profile removes that deliberate difference along with
the errors. Its low-level water vapour goes from 11.50 to 13.23 g/kg. So this section is not
asking what a bug was worth. It is asking what the design choice was worth.

Three mechanisms could raise entrainment efficiency, and they are what the diagnostics below
are built to separate.

1. **Updraft speed.** Stronger updrafts carry more of whatever they ingest.
2. **The virtual-temperature effect of the added moisture.** More vapour means more buoyancy,
   a lower lifting condensation level, and a longer-lived storm.
3. **Geometry.** Where the updraft roots, and therefore which air it draws from.

The three are not independent, and the honest question is not which one is true. It is which
one the tracers actually respond to.

### The second identity

§2.9 split the in-updraft rain-sourced mixing ratio into supply per unit updraft mass and the
fraction of supply captured. That split cannot answer this question, because both of its
factors are blind to what the rest of the boundary layer looks like. A second exact split can.
Write $N$ for tracer inside the updraft, $A$ for cumulative emission, and $M$ for updraft air
mass, with subscripts $RS$ for rain-sourced and $FS$ for fixed-source. Then

$$ q_{RS} \;=\; \underbrace{\frac{N_{FS}}{M}}_{\text{general lofting}} \;\times\; \underbrace{\frac{A_{RS}}{A_{FS}}}_{\text{emitted RS share}} \;\times\; \underbrace{\frac{N_{RS}/N_{FS}}{A_{RS}/A_{FS}}}_{\text{selectivity}} $$

The middle factor is a property of the rain field alone, because the fixed-source tracers are
released on a schedule no storm can influence. So $A_{FS}$ is the same number in both arms, and
that factor says only how much of the boundary layer has been rained on.

The last factor is the one that matters here. Selectivity asks whether the updraft draws
*preferentially* from air that has sat under rain. A value of one means it samples the boundary
layer indiscriminately. A value above one means it roots in the rained-on part.

In [ ]:
from importlib import reload

reload(hu)

# The storm this section is about. Everything else is inherited from §2.9's parameters.
ENTRAINMENT_IC_STORM_KEY = "ic_wk"

print(f"Storm:  {blt.STORM_LABELS[ENTRAINMENT_IC_STORM_KEY]}")
print(f"Tracer: {ENTRAINMENT_RS_TRACER}")

### 2.10.1 First, remove the longevity confound

The entrainment efficiency $N/A$ divides what is in the updraft *now* by everything emitted
*ever*. A storm that dies therefore scores an efficiency approaching zero, whatever it did
while it was alive. The original isolated convection dies: its updraft mass falls by four
orders of magnitude between $t = 80$ and $t = 180$ min, while the harmonized storm is still
running at the end of the simulation.

That difference alone could produce the whole result, so it has to be ruled out before any
mechanism is proposed. The table below recomputes every ratio over four windows. The 60–120
minute window is the control: both storms are healthy in it. A ratio that holds up there is
about how the storms behave. A ratio that only appears in the late windows is about how long
they last.

In [ ]:
# Every headline ratio, over four time windows. The supercell rows are the arithmetic check
# and must be all ones.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    display(HTML(f"<h4>{blt.STORM_LABELS[storm_key]}</h4>"))
    display(
        hu.entrainment_window_sensitivity_table(
            tracer_totals_dss_by_arm,
            storm_key,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
        ).round(3)
    )

### 2.10.2 The selectivity split

Each table below is the three-factor identity from the section header. The three log ratios sum
to the fourth, and the helper asserts that they do.

The row to read is the last-but-one. General lofting says whether the updraft carries more
low-level air of any kind. If it does not, then updraft speed cannot be the mechanism, because
a faster updraft would carry the fixed-source boundary-layer tracer up as readily as the
rain-sourced one.

In [ ]:
# The second identity, over the full run and over the both-storms-alive window.
for window_label, analysis_window_minutes in [
    ("full simulation", (0.0, np.inf)),
    ("60-120 min (both storms alive)", (60.0, 120.0)),
]:
    display(
        HTML(
            f"<h4>{blt.STORM_LABELS[ENTRAINMENT_IC_STORM_KEY]} &mdash;"
            f" {window_label}</h4>"
        )
    )
    display(
        hu.rain_marking_selectivity_table(
            tracer_totals_dss_by_arm,
            ENTRAINMENT_IC_STORM_KEY,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
            analysis_window_minutes=analysis_window_minutes,
        )
    )

# All three storms side by side, ratio column only. The squall line is the contrast case: its
# decline sits in the first two factors, not in selectivity.
display(HTML("<h4>All storms, ratio column (full simulation)</h4>"))
display(
    pd.DataFrame({
        blt.STORM_LABELS[storm_key]: hu.rain_marking_selectivity_table(
            tracer_totals_dss_by_arm,
            storm_key,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
        )["ratio"]
        for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER
    }).round(3)
)

In [ ]:
# The three factors and their product against time. A table cannot show that one factor is
# steady while another grows, and selectivity is expected to grow through a storm's life:
# rain has to fall before there is anything rain-marked to draw from.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_rain_marking_selectivity(
        tracer_totals_dss_by_arm,
        storm_key,
        rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
    )
    cm.plotting.fig_multisave(
        fig, f"rain_marking_selectivity__{storm_key}", FIGURE_OUTPUT_DIR
    )

### 2.10.3 Testing the three candidate mechanisms

The updraft's own shape is what discriminates the three candidates, and the tracer profile
dataset already carries it: it counts in-updraft gridpoints at every level, so no extra pass
over the raw output is needed.

`blt` defines an updraft point as cloudy **and** rising. The base of the diagnosed updraft is
therefore close to cloud base, which is what makes it sensitive to the moisture change. A run
with more low-level vapour has a lower lifting condensation level, a lower cloud base, and so
an updraft that reaches further down into the layer the rain-sourced tracer is emitted into.

**One caveat on the base panel.** A storm that is dying also shows a rising base, because only
elevated cloudy remnants are left. So the middle panel of the figure conflates cloud base with
storm stage, and the 60–120 minute window in §2.10.1 is the control for that.

The last cell attributes the change in entrained tracer to the two things it can come from. The
count is a sum over levels of mass times mixing ratio, so a change in it is either the updraft
occupying more mass at a given height, or the air at that height being richer. The Bennet
decomposition splits it exactly, and averaging the two arms inside each term is what removes
the interaction remainder that a one-sided attribution would leave behind.

In [ ]:
# Updraft shape, over the mature storm.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    display(HTML(f"<h4>{blt.STORM_LABELS[storm_key]}</h4>"))
    display(
        hu.updraft_geometry_table(
            tracer_totals_profile_dss_by_arm,
            storm_key,
            analysis_window_minutes=MATURE_STORM_WINDOW_MINUTES,
        ).T.round(1)
    )

In [ ]:
# Where the updraft is, how far down it reaches, and how long it lasts. Updraft *speed* is
# deliberately absent from this figure: if these three panels move a long way, speed is not
# what needs explaining. The peak-updraft panel in §2.9's rain figure covers speed.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    fig, axs = hu.plot_updraft_geometry(
        tracer_totals_profile_dss_by_arm,
        storm_key,
        analysis_window_minutes=MATURE_STORM_WINDOW_MINUTES,
    )
    cm.plotting.fig_multisave(
        fig, f"updraft_geometry__{storm_key}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# Was the extra entrained tracer carried by an updraft that occupies more volume, or by air
# that is richer at the same height? The layer breakdown is what makes the answer readable:
# a volume term concentrated low down means the updraft started reaching into air it did not
# previously touch.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    display(HTML(f"<h4>{blt.STORM_LABELS[storm_key]}</h4>"))
    display(
        hu.entrained_tracer_shift_share_table(
            tracer_totals_profile_dss_by_arm,
            storm_key,
            ORIGINAL_ARM_LABEL,
            rain_sourced_tracer=ENTRAINMENT_RS_TRACER,
            analysis_window_minutes=MATURE_STORM_WINDOW_MINUTES,
        ).round(2)
    )

### 2.10.4 What this section establishes

The numbers below come from the run of this notebook that produced the cells above. Re-run it
and re-read them.

**It is not updraft speed.** The peak updraft does rise, from 24.9 to 32.2 m/s, and the
mature-phase mean of it doubles. But general lofting *falls*: the boundary-layer tracer per
kilogram of updraft comes out at 0.73 of the original over the full run. A faster updraft would
carry the fixed-source boundary-layer tracer up as readily as the rain-sourced tracer. It does
not. The harmonized storm's updraft is not carrying more low-level air. It is carrying
differently-sourced low-level air.

**It is not only longevity.** The original storm dies and the harmonized one does not, which
inflates every late-window ratio. In the 60–120 minute window, when both storms are healthy and
their updraft masses differ by only 1.38, the selectivity ratio is still 1.91 and the
rain-sourced mixing ratio ratio is still 2.65. The effect survives the control.

**The three-factor split.** Over the full run the rain-sourced mixing ratio ratio of 2.34 is

$$ 2.34 \;=\; \underbrace{0.73}_{\text{general lofting}} \times \underbrace{1.65}_{\text{emitted RS share}} \times \underbrace{1.94}_{\text{selectivity}} $$

Roughly half the gain is that more of the boundary layer has been rained on. The other half is
that the updraft draws about twice as preferentially from the part that has.

**Where the selectivity comes from: geometry.** Over the mature storm the median updraft base
drops from 2834 m to 1208 m, and the lowest level the updraft ever reaches drops from 655 m to
312 m. The updraft area below 2 km rises by a factor of 5. The updraft *top* barely moves,
13.3 km to 14.1 km, and the mass-weighted mean height falls slightly. So the storm did not get
deeper. It got rooted lower, into the layer where the rain-sourced tracer is emitted.

That is the moisture change acting through cloud base. Low-level vapour rises from 11.50 to
13.23 g/kg, and the lifting condensation level falls 381 m, from 1734 m to 1353 m (§1.7).
Because a point counts as updraft only if it is cloudy, a lower condensation level lowers the
diagnosed updraft base directly.

**The shift-share agrees.** Of the extra entrained rain-sourced tracer, 55% is attributable to
the updraft occupying more mass and 45% to the air being richer at a given height. The volume
term is concentrated in the 1.5–3 km layer, which contributes 20% of the whole change on its
own. That layer is where the harmonized updraft exists and the original one, in its mature
phase, does not.

**The answer to the question.** The efficiency gain is a geometry effect produced by the
moisture increase, not an updraft-speed effect. The virtual-temperature contribution of the
extra vapour is the upstream cause rather than a separate mechanism: it raises CAPE, it lowers
cloud base, and it keeps the storm alive longer. What the tracers respond to is the second of
those three.

**Contrast with the squall line.** §2.9 found the squall line's decline in the first two
factors and in general lofting, with selectivity roughly unchanged. The isolated convection's
gain is almost entirely in the third factor. The two storms move in opposite directions for
unrelated reasons, which is the case for treating the cross-storm ratios of §2.6, rather than
either storm's absolute numbers, as the quantities the manuscript should quote.

# Conclusions

*Fill this in once Part 2 has been run.*

**Part 1 (initial state) — established:**

- How far the original three setups actually sit from each other, split by height window, in
  §1.2. The squall-line row is the informative one: it and the supercell were designed to be
  thermodynamically identical, so every non-zero entry there is unintended.
- How much of that this harmonization removes, in §1.3, and the residual that remains.
- Which of those differences are the pressure anchoring alone, in §1.4: anything that collapses
  on a pressure vertical coordinate was the anchoring, anything that survives it was not.
- Where in the column the convective potential moved, in §1.5. The per-storm skew-Ts shade the
  CAPE *change* directly, so a change concentrated in the low levels can be told apart from one
  spread through the depth of the troposphere — a distinction no scalar CAPE can make.
- The surface-based CAPE gap between the squall line and supercell in §1.7, before and after
  — the single cleanest scalar measure of how unfair the original comparison was.

**Part 2 (results) — to fill in after running:**

- Whether each storm's headline numbers moved (§2.6, first table).
- **Whether the ratios between storms moved** (§2.6, second table). This is the answer to the
  question the notebook was written for. If the cross-storm ratios are stable, the
  manuscript's comparative conclusions are not artefacts of the setup differences, regardless
  of how much the individual numbers shifted.
- Whether the coarse mass-flux estimator's bias is common-mode across storms (§2.4.1). If it
  is not, treat the Figure 11 rows 1–2 and Figure 12 comparisons as qualitative.
- Whether the fixed right-mover mask still fits the rerun supercell (§2.7).
- What the storms actually *look* like in each arm (§2.8) — the animations are the fastest
  way to tell a change in storm intensity from a change in storm mode.

**Cross-reference:** the companion notebook for the other harmonization is
`harmonized_on_sl_profile.ipynb`. A
conclusion that holds in both is much stronger than one that holds in only one, because the
two harmonizations equalize the storms in opposite directions — B gives all three the *correct* profile, A gives them all the same *erroneous* one.